In [1]:
# =============================================================================
# FİNANSAL METİNLERDE DUYGU ANALİZİ
# =============================================================================

In [2]:
# =============================================================================
# 0. KURULUM
# =============================================================================

!pip install -q pandas numpy matplotlib seaborn nltk scikit-learn transformers torch keybert statsmodels wordcloud sentence-transformers optuna lime joblib scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 23.3 MB/s eta 0:00:00


In [3]:
# =============================================================================
# 1. DRIVE BAĞLAMA
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# =============================================================================
# 2. KÜTÜPHANELER
# =============================================================================

import os
import re
import time
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns


def show_plot():
    if matplotlib.get_backend().lower() != 'agg':
        plt.show()

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib

# Ağır üçüncü taraf paketleri isteğe bağlı yükleme için kontrol
SKIP_HEAVY = False  # True ise KeyBERT, sentence-transformers, transformers gibi paketler atlanır

try:
    if not SKIP_HEAVY:
        from keybert import KeyBERT
    else:
        KeyBERT = None
except Exception:
    KeyBERT = None

from wordcloud import WordCloud

try:
    if not SKIP_HEAVY:
        from sentence_transformers import SentenceTransformer
    else:
        SentenceTransformer = None
except Exception:
    SentenceTransformer = None

from statsmodels.stats.contingency_tables import mcnemar
import lime.lime_text as lime_text

import torch
from torch.optim import AdamW

try:
    if not SKIP_HEAVY:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
    else:
        AutoTokenizer = None
        AutoModelForSequenceClassification = None
except Exception:
    AutoTokenizer = None
    AutoModelForSequenceClassification = None

try:
    import optuna
except ImportError:
    optuna = None


In [8]:
# =============================================================================
# 3. GENEL AYARLAR
# =============================================================================

from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive")

PHRASEBANK_PATH = BASE_DIR / "all-data.csv"
TWITTER_PATH = BASE_DIR / "twitter_financial_news.csv"
FIQA_PATH = BASE_DIR / "fiqa_sentiment.csv"

OUTPUT_DIR = BASE_DIR / "proje_ciktilari"

for klasor in ["grafikler", "metrikler", "modeller", "tablolar"]:
    (OUTPUT_DIR / klasor).mkdir(parents=True, exist_ok=True)

print("PhraseBank:", PHRASEBANK_PATH)
print("Twitter:", TWITTER_PATH)
print("FiQA:", FIQA_PATH)

# =============================================================================
# GLOBAL SABİTLER
# =============================================================================

VALID_LABELS = [
    "negative",
    "neutral",
    "positive"
]

RANDOM_STATE = 42

PhraseBank: /content/drive/MyDrive/all-data.csv
Twitter: /content/drive/MyDrive/twitter_financial_news.csv
FiQA: /content/drive/MyDrive/fiqa_sentiment.csv


In [9]:
# =============================================================================
# 4. ÜÇ VERİ SETİNİN YÜKLENMESİ VE BİRLEŞTİRİLMESİ
# =============================================================================

print("\n[1] Üç veri seti yükleniyor ve birleştiriliyor...")

required_files = {
    "Financial PhraseBank": PHRASEBANK_PATH,
    "Twitter Financial News": TWITTER_PATH,
    "FiQA Sentiment": FIQA_PATH,
}

missing_files = []

for name, path in required_files.items():
    if not path.exists():
        missing_files.append(str(path))

if missing_files:
    print("Eksik dosyalar:")
    for file in missing_files:
        print("-", file)

    raise FileNotFoundError("Veri seti dosyaları bulunamadı.")


def prepare_dataset(df, dataset_name):
    df = df.copy()
    df.columns = [col.lower().strip() for col in df.columns]

    if "sentence" in df.columns and "text" not in df.columns:
        df = df.rename(columns={"sentence": "text"})

    if "sentiment" in df.columns and "label" not in df.columns:
        df = df.rename(columns={"sentiment": "label"})

    if dataset_name == "FiQA Sentiment Classification":
        if "score" not in df.columns:
            raise ValueError("FiQA dosyasında 'score' sütunu bulunamadı.")

        df["score"] = pd.to_numeric(df["score"], errors="coerce")

        def score_to_label(score):
            if pd.isna(score):
                return None
            elif score < 0:
                return "negative"
            elif score > 0:
                return "positive"
            else:
                return "neutral"

        df["label"] = df["score"].apply(score_to_label)

    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError(
            f"{dataset_name} dosyasında text ve label sütunları bulunamadı. "
            f"Mevcut sütunlar: {df.columns.tolist()}"
        )

    df = df[["text", "label"]].dropna()

    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.lower().str.strip()

    label_map = {
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive",
        "bearish": "negative",
        "bullish": "positive",
        "0": "negative",
        "1": "neutral",
        "2": "positive",
        0: "negative",
        1: "neutral",
        2: "positive",
    }

    df["label"] = df["label"].map(label_map)

    df = df.dropna(subset=["label"])
    df = df[df["text"] != ""]
    df["dataset"] = dataset_name

    return df[["dataset", "text", "label"]]


df_phrasebank_raw = pd.read_csv(
    PHRASEBANK_PATH,
    encoding="latin-1",
    names=["label", "text"]
)

df_twitter_raw = pd.read_csv(TWITTER_PATH)
df_fiqa_raw = pd.read_csv(FIQA_PATH)

df_phrasebank = prepare_dataset(df_phrasebank_raw, "Financial PhraseBank")
df_twitter = prepare_dataset(df_twitter_raw, "Twitter Financial News Sentiment")
df_fiqa = prepare_dataset(df_fiqa_raw, "FiQA Sentiment Classification")

df = pd.concat(
    [df_phrasebank, df_twitter, df_fiqa],
    ignore_index=True
)

df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("\nBirleştirilmiş veri seti hazır.")
print(f"Toplam örnek sayısı: {len(df)}")

print("\nVeri setlerine göre dağılım:")
print(df["dataset"].value_counts())

print("\nSınıflara göre dağılım:")
print(df["label"].value_counts().reindex(VALID_LABELS))


[1] Üç veri seti yükleniyor ve birleştiriliyor...

Birleştirilmiş veri seti hazır.
Toplam örnek sayısı: 17890

Veri setlerine göre dağılım:
dataset
Twitter Financial News Sentiment    11928
Financial PhraseBank                 4840
FiQA Sentiment Classification        1122
Name: count, dtype: int64

Sınıflara göre dağılım:
label
negative    2776
neutral     5283
positive    9831
Name: count, dtype: int64


In [10]:
# =============================================================================
# 5. VERİ SETİ DAĞILIM GRAFİKLERİ
# =============================================================================

class_dist = df["label"].value_counts().reindex(VALID_LABELS)

plt.figure(figsize=(8, 5))
sns.barplot(x=class_dist.index, y=class_dist.values)
plt.title("Combined Dataset - Class Distribution")
plt.xlabel("Sentiment Class")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grafikler" / "sinif_dagilimi.png", dpi=150)
show_plot()

dataset_dist = df["dataset"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=dataset_dist.values, y=dataset_dist.index)
plt.title("Sample Distribution by Dataset")
plt.xlabel("Number of Samples")
plt.ylabel("Dataset")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grafikler" / "veri_seti_dagilimi.png", dpi=150)
show_plot()

plt.figure(figsize=(7, 7))
plt.pie(class_dist.values, labels=class_dist.index, autopct="%1.1f%%", startangle=90)
plt.title("Class Ratios in Combined Dataset")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grafikler" / "sinif_oranlari_pasta.png", dpi=200)
show_plot()

In [11]:
# =============================================================================
# 6. KELİME UZUNLUĞU ANALİZİ
# =============================================================================

print("\n[2] Kelime uzunluğu analizi yapılıyor...")

df["word_count"] = df["text"].apply(lambda x: len(x.split()))

word_stats = df["word_count"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print(word_stats)

word_stats.to_csv(
    OUTPUT_DIR / "tablolar" / "kelime_uzunlugu_istatistikleri.csv"
)

plt.figure(figsize=(9, 5))
sns.histplot(df["word_count"], bins=30, kde=True)
plt.title("Word Count Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grafikler" / "kelime_uzunlugu_dagilimi.png", dpi=150)
show_plot()

plt.figure(figsize=(9, 4))
sns.boxplot(x=df["word_count"])
plt.title("Word Count Boxplot")
plt.xlabel("Number of Words")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grafikler" / "kelime_uzunlugu_boxplot.png", dpi=200)
show_plot()

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="label", y="word_count", order=VALID_LABELS)
plt.title("Word Count by Sentiment Class")
plt.xlabel("Sentiment Class")
plt.ylabel("Number of Words")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "sinif_bazli_kelime_uzunlugu_boxplot.png",
    dpi=200
)
show_plot()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="dataset", y="word_count")
plt.title("Word Count by Dataset")
plt.xlabel("Dataset")
plt.ylabel("Number of Words")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "veri_seti_bazli_kelime_uzunlugu_boxplot.png",
    dpi=200
)
show_plot()

p99_words = int(np.ceil(df["word_count"].quantile(0.99)))
MAX_LENGTH = 64 if p99_words <= 55 else 96

print(f"\nFinBERT max_length seçimi: {MAX_LENGTH} (p99={p99_words})")



[2] Kelime uzunluğu analizi yapılıyor...
count    17890.000000
mean        15.134544
std          8.156037
min          1.000000
50%         13.000000
75%         19.000000
90%         25.000000
95%         32.000000
99%         44.000000
max         81.000000
Name: word_count, dtype: float64

FinBERT max_length seçimi: 64 (p99=44)


In [12]:
# =============================================================================
# 7. METİN TEMİZLEME
# =============================================================================

import re
import nltk

print("\n[3] Metin temizleme uygulanıyor...")

# Stopwords yükle
try:
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words("english"))
except:
    nltk.download("stopwords")
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words("english"))

# Duygu analizinde önemli olan olumsuzluk kelimelerini koru
KEEP_NEGATIVES = {
    "not",
    "no",
    "never",
    "n't",
    "neither",
    "nor",
    "none",
    "nothing"
}

STOPWORDS = STOPWORDS - KEEP_NEGATIVES


def clean_text_basic(text):
    """
    TF-IDF tabanlı modeller için temel metin temizleme.
    NLTK tokenizer yerine split() kullanılır.
    Böylece punkt / punkt_tab hataları oluşmaz.
    """

    text = str(text).lower()

    # Fazla boşlukları temizle
    text = re.sub(r"\s+", " ", text)

    # Noktalama işaretlerini kaldır
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Tokenizasyon
    words = text.split()

    # Stopword temizliği
    words = [
        w
        for w in words
        if (
            w.isalnum()
            and w not in STOPWORDS
            and len(w) > 1
        )
    ]

    return " ".join(words)


# Temizlenmiş metin oluştur
df["clean_text"] = df["text"].apply(clean_text_basic)

# Boş kalan satırları kaldır
df = df[
    df["clean_text"].str.strip() != ""
].reset_index(drop=True)

print(f"Temizleme sonrası veri boyutu: {df.shape}")

# Örnek kayıtlar
cleaning_examples = df[
    [
        "dataset",
        "text",
        "clean_text",
        "label",
        "word_count"
    ]
].head(10)

# Dosyaya kaydet
try:
    cleaning_examples.to_csv(
        OUTPUT_DIR / "tablolar" / "temizleme_ornekleri.csv",
        index=False
    )
except:
    pass

print("\nTemizleme örnekleri:")
print(cleaning_examples)



[3] Metin temizleme uygulanıyor...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Temizleme sonrası veri boyutu: (17889, 5)

Temizleme örnekleri:
                dataset                                               text  \
0  Financial PhraseBank  According to Gran , the company has no plans t...   
1  Financial PhraseBank  Technopolis plans to develop in stages an area...   
2  Financial PhraseBank  The international electronic industry company ...   
3  Financial PhraseBank  With the new production plant the company woul...   
4  Financial PhraseBank  According to the company 's updated strategy f...   
5  Financial PhraseBank  FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...   
6  Financial PhraseBank  For the last quarter of 2010 , Componenta 's n...   
7  Financial PhraseBank  In the third quarter of 2010 , net sales incre...   
8  Financial PhraseBank  Operating profit rose to EUR 13.1 mn from EUR ...   
9  Financial PhraseBank  Operating profit totalled EUR 21.1 mn , up fro...   

                                          clean_text     label  word_count  


In [13]:
# =============================================================================
# 8. N-GRAM, KELİME BULUTU VE TF-IDF TERİMLERİ
# =============================================================================

print("\n[4] Metin madenciliği analizleri yapılıyor...")


def plot_top_ngrams(corpus, ngram_range=(2, 3), top_n=15, filename="ngram_analizi.png"):
    vec = CountVectorizer(ngram_range=ngram_range, max_features=5000)
    X = vec.fit_transform(corpus)

    total_counts = np.asarray(X.sum(axis=0)).ravel()
    terms = vec.get_feature_names_out()

    top_indices = total_counts.argsort()[::-1][:top_n]

    ngram_df = pd.DataFrame({
        "ngram": terms[top_indices],
        "frequency": total_counts[top_indices]
    })

    ngram_df.to_csv(
        OUTPUT_DIR / "tablolar" / filename.replace(".png", ".csv"),
        index=False
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=ngram_df, x="frequency", y="ngram")
    plt.title("Most Frequent 2-3 Word N-grams")
    plt.xlabel("Frequency")
    plt.ylabel("N-gram")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "grafikler" / filename, dpi=150)
    show_plot()

    return ngram_df


ngram_df = plot_top_ngrams(df["clean_text"], ngram_range=(2, 3), top_n=15)

print("\nEn sık n-gram örnekleri:")
print(ngram_df.head())

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, label in zip(axes, VALID_LABELS):
    class_text = " ".join(df.loc[df["label"] == label, "clean_text"])

    if class_text.strip():
        wc = WordCloud(
            width=500,
            height=400,
            background_color="white",
            max_words=60
        ).generate(class_text)

        ax.imshow(wc, interpolation="bilinear")

    ax.set_title(f"{label.capitalize()} Texts")
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grafikler" / "kelime_bulutu_sinif_bazli.png", dpi=150)
show_plot()

tfidf_temp = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_all_tfidf = tfidf_temp.fit_transform(df["clean_text"])

feature_names = tfidf_temp.get_feature_names_out()
y_all = df["label"].values

all_tfidf_terms = []

for label in VALID_LABELS:
    mask = y_all == label
    mean_scores = X_all_tfidf[mask].mean(axis=0).A1
    top_indices = np.argsort(mean_scores)[-15:][::-1]

    print(f"\n{label.upper()} sınıfında en yüksek TF-IDF terimleri:")

    for i in top_indices:
        print(f"  {feature_names[i]:30s} {mean_scores[i]:.4f}")

        all_tfidf_terms.append({
            "label": label,
            "term": feature_names[i],
            "mean_tfidf": mean_scores[i]
        })

pd.DataFrame(all_tfidf_terms).to_csv(
    OUTPUT_DIR / "tablolar" / "sinif_bazli_tfidf_terimleri.csv",
    index=False
)

for label in VALID_LABELS:
    plot_df = pd.DataFrame(all_tfidf_terms)
    plot_df = plot_df[plot_df["label"] == label].sort_values(
        "mean_tfidf",
        ascending=True
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=plot_df, x="mean_tfidf", y="term")
    plt.title(f"Top TF-IDF Terms for {label.capitalize()} Class")
    plt.xlabel("Mean TF-IDF Score")
    plt.ylabel("Term")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "grafikler" / f"tfidf_terimleri_{label}.png", dpi=200)
    show_plot()



[4] Metin madenciliği analizleri yapılıyor...

En sık n-gram örnekleri:
                        ngram  frequency
0                    https co       6512
1     marketscreener https co        591
2        marketscreener https        591
3  stock marketscreener https        396
4        stock marketscreener        396

NEGATIVE sınıfında en yüksek TF-IDF terimleri:
  co                             0.0393
  https                          0.0385
  https co                       0.0385
  eur                            0.0171
  stock                          0.0139
  mn                             0.0136
  misses                         0.0129
  sales                          0.0129
  profit                         0.0117
  lower                          0.0108
  year                           0.0108
  oil                            0.0106
  shares                         0.0098
  market                         0.0098
  china                          0.0096

NEUTRAL sınıfında en yüksek TF-I

In [14]:
# =============================================================================
# 9. KEYBERT ANAHTAR KELİME ÇIKARIMI VE VADER SKORU
# =============================================================================

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

print("\n[5] KeyBERT ile anahtar kelime çıkarılıyor...")

# VADER sözlüğünü indir
nltk.download("vader_lexicon", quiet=True)

# KeyBERT modeli (isteğe bağlı)
kw_model = None
if KeyBERT is not None:
    try:
        kw_model = KeyBERT("sentence-transformers/all-MiniLM-L6-v2")
    except Exception as e:
        print("KeyBERT yüklenemedi:", e)
        kw_model = None


def extract_keywords_with_scores(text: str, top_n: int = 3):
    if kw_model is None:
        return []

    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=0.50,
        )
        return keywords
    except Exception:
        return []


def only_keyword_text(keyword_score_list):
    return " ".join([kw for kw, score in keyword_score_list])


# KeyBERT 17.890 satırda uzun sürebilir.
df["keyword_score_pairs"] = df["text"].apply(
    lambda x: extract_keywords_with_scores(x, top_n=3)
)

df["keywords"] = df["keyword_score_pairs"].apply(
    lambda pairs: [kw for kw, score in pairs]
)

df["keyword_text"] = df["keyword_score_pairs"].apply(only_keyword_text)

# VADER duygu skoru
sia = SentimentIntensityAnalyzer()

df["vader_score"] = df["text"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

print("keyword_text ve vader_score sütunları oluşturuldu.")
print(df[["text", "keyword_text", "vader_score"]].head())



[5] KeyBERT ile anahtar kelime çıkarılıyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

keyword_text ve vader_score sütunları oluşturuldu.
                                                text  \
0  According to Gran , the company has no plans t...   
1  Technopolis plans to develop in stages an area...   
2  The international electronic industry company ...   
3  With the new production plant the company woul...   
4  According to the company 's updated strategy f...   

                                      keyword_text  vader_score  
0             production russia gran company plans      -0.1280  
1   technopolis plans develop stages square meters      -0.2960  
2        company elcoteq layoffs employees tallinn       0.0000  
3  increase production plant company raw materials       0.8555  
4               basware sales growth profit margin       0.6705  


In [15]:
# =============================================================================
# 10. EĞİTİM / TEST AYRIMI
# =============================================================================

from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
VALID_LABELS = ["negative", "neutral", "positive"]

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

# Colab uyumlu
BASE_DIR = Path("/content/drive/MyDrive")

OUTPUT_DIR = BASE_DIR / "proje_ciktilari"

for folder in ["grafikler", "metrikler", "modeller", "tablolar"]:
    (OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)

print("\n[6] Eğitim/test ayrımı yapılıyor...")

X = df[["dataset", "text", "clean_text", "keyword_text", "vader_score"]].copy()
y = df["label"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

y_train_ids = y_train.map(label2id).to_numpy()
y_test_ids = y_test.map(label2id).to_numpy()

print(f"Eğitim örneği: {len(X_train)}")
print(f"Test örneği: {len(X_test)}")


[6] Eğitim/test ayrımı yapılıyor...
Eğitim örneği: 14311
Test örneği: 3578


In [16]:
# =============================================================================
# 11. BASELINE MODELLER: TF-IDF + LR / SVM / ENSEMBLE
# =============================================================================

import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

try:
    import optuna
except:
    optuna = None

RUN_OPTUNA = False

print("\n[7] Baseline modeller eğitiliyor...")

baseline_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = baseline_tfidf.fit_transform(X_train["clean_text"])
X_test_tfidf = baseline_tfidf.transform(X_test["clean_text"])

lr = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    random_state=RANDOM_STATE
)

lr.fit(X_train_tfidf, y_train)
lr_preds = lr.predict(X_test_tfidf)

svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=3000,
    dual=False,
    random_state=RANDOM_STATE
)

svm.fit(X_train_tfidf, y_train)
svm_preds = svm.predict(X_test_tfidf)

ensemble = VotingClassifier(
    estimators=[
        ("lr", lr),
        ("svm", svm)
    ],
    voting="hard"
)

ensemble.fit(X_train_tfidf, y_train)
ens_preds = ensemble.predict(X_test_tfidf)

joblib.dump(
    baseline_tfidf,
    OUTPUT_DIR / "modeller" / "baseline_tfidf_vectorizer.joblib"
)

joblib.dump(
    lr,
    OUTPUT_DIR / "modeller" / "lr_model.joblib"
)

joblib.dump(
    svm,
    OUTPUT_DIR / "modeller" / "svm_model.joblib"
)

joblib.dump(
    ensemble,
    OUTPUT_DIR / "modeller" / "ensemble_lr_svm.joblib"
)

if RUN_OPTUNA and optuna is not None:
    print("\nOptuna ile SVM hiperparametre optimizasyonu...")

    def objective(trial):
        C = trial.suggest_float("C", 0.1, 10.0, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        gamma = "scale"

        model = SVC(
            C=C,
            kernel=kernel,
            gamma=gamma,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )

        score = cross_val_score(
            model,
            X_train_tfidf,
            y_train,
            cv=3,
            scoring="f1_macro"
        ).mean()

        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=10, show_progress_bar=False)

    print("En iyi SVM parametreleri:", study.best_params)

    with open(
        OUTPUT_DIR / "metrikler" / "optuna_svm_sonucu.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(str(study.best_params) + "\n")
        f.write(f"Best F1-macro: {study.best_value:.4f}\n")

print("Baseline modeller başarıyla eğitildi.")



[7] Baseline modeller eğitiliyor...
Baseline modeller başarıyla eğitildi.


In [17]:
# =============================================================================
# 12. ÖNERİLEN HİBRİT MODEL
# =============================================================================

print("\n[8] Önerilen hibrit model eğitiliyor...")

main_text_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

keyword_vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    min_df=1
)

vader_scaler = StandardScaler(with_mean=False)

X_train_main = main_text_vectorizer.fit_transform(X_train["clean_text"])
X_test_main = main_text_vectorizer.transform(X_test["clean_text"])

kw_train_texts = X_train["keyword_text"].fillna("").astype(str)
kw_test_texts = X_test["keyword_text"].fillna("").astype(str)

try:
    X_train_kw = keyword_vectorizer.fit_transform(kw_train_texts)
    X_test_kw = keyword_vectorizer.transform(kw_test_texts)
except ValueError:
    # Boş sözlük hatası (tüm dokümanlar stopword veya boş). Boş sütun matrisi kullan.
    n_train = len(X_train)
    n_test = len(X_test)

    X_train_kw = csr_matrix((n_train, 0))
    X_test_kw = csr_matrix((n_test, 0))

    # vectorizer kullanılamıyor
    keyword_vectorizer = None

X_train_vader = csr_matrix(
    vader_scaler.fit_transform(X_train[["vader_score"]])
)

X_test_vader = csr_matrix(
    vader_scaler.transform(X_test[["vader_score"]])
)

X_train_hybrid = hstack([
    X_train_main,
    X_train_kw,
    X_train_vader
])

X_test_hybrid = hstack([
    X_test_main,
    X_test_kw,
    X_test_vader
])

hybrid_model = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    random_state=RANDOM_STATE
)

hybrid_model.fit(X_train_hybrid, y_train)
hybrid_preds = hybrid_model.predict(X_test_hybrid)

joblib.dump(
    main_text_vectorizer,
    OUTPUT_DIR / "modeller" / "hybrid_main_text_vectorizer.joblib"
)

joblib.dump(
    keyword_vectorizer,
    OUTPUT_DIR / "modeller" / "hybrid_keyword_vectorizer.joblib"
)

joblib.dump(
    vader_scaler,
    OUTPUT_DIR / "modeller" / "hybrid_vader_scaler.joblib"
)

joblib.dump(
    hybrid_model,
    OUTPUT_DIR / "modeller" / "onerilen_hibrit_model.joblib"
)


[8] Önerilen hibrit model eğitiliyor...


['/content/drive/MyDrive/proje_ciktilari/modeller/onerilen_hibrit_model.joblib']

In [18]:
# =============================================================================
# 13. FINBERT FINE-TUNE
# =============================================================================

bert_preds = None
finbert_model = None
finbert_tokenizer = None


def train_and_evaluate_finbert():
    print("\n[9] FinBERT fine-tune eğitimi başlıyor...")

    model_name = "ProsusAI/finbert"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    train_enc = tokenizer(
        X_train["text"].tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

    test_enc = tokenizer(
        X_test["text"].tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

    class SentimentDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {
                key: torch.tensor(val[idx])
                for key, val in self.encodings.items()
            }

            item["labels"] = torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )

            return item

        def __len__(self):
            return len(self.labels)

    train_ds = SentimentDataset(train_enc, y_train_ids)
    test_ds = SentimentDataset(test_enc, y_test_ids)

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=32,
        shuffle=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Çalışılan cihaz: {device}")

    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    epochs = 3
    start = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Ortalama kayıp: {total_loss / len(train_loader):.4f}"
        )

    print(
        f"FinBERT eğitimi tamamlandı. "
        f"Süre: {time.time() - start:.1f} saniye"
    )

    model.save_pretrained(OUTPUT_DIR / "modeller" / "finbert")
    tokenizer.save_pretrained(OUTPUT_DIR / "modeller" / "finbert")

    model.eval()

    all_preds = []

    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop("labels")

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            preds = torch.argmax(
                outputs.logits,
                dim=1
            ).cpu().numpy()

            all_preds.extend(preds)

    return np.array(all_preds), model, tokenizer, device



RUN_FINBERT = True

if RUN_FINBERT:
    if AutoTokenizer is None or AutoModelForSequenceClassification is None:
        print("\n[9] FinBERT atlandı: transformers paketi yüklenmemiş veya SKIP_HEAVY=True.")
    else:
        bert_preds, finbert_model, finbert_tokenizer, finbert_device = train_and_evaluate_finbert()
else:
    print("\n[9] FinBERT fine-tune RUN_FINBERT=False olduğu için atlandı.")



[9] FinBERT fine-tune eğitimi başlıyor...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Çalışılan cihaz: cuda
Epoch 1/3 - Ortalama kayıp: 0.5758
Epoch 2/3 - Ortalama kayıp: 0.3017
Epoch 3/3 - Ortalama kayıp: 0.1567
FinBERT eğitimi tamamlandı. Süre: 519.7 saniye


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
# =============================================================================
# 14. MODEL KARŞILAŞTIRMA
# =============================================================================

print("\n[10] Model karşılaştırması yapılıyor...")

model_predictions = {
    "TF-IDF + Logistic Regression": np.array([label2id[p] for p in lr_preds]),
    "TF-IDF + Linear SVM": np.array([label2id[p] for p in svm_preds]),
    "Ensemble LR+SVM": np.array([label2id[p] for p in ens_preds]),
    "Önerilen Hibrit Model": np.array([label2id[p] for p in hybrid_preds]),
}

if bert_preds is not None:
    model_predictions["FinBERT Fine-Tuned"] = bert_preds

results = []

for model_name, preds in model_predictions.items():
    acc = accuracy_score(y_test_ids, preds)
    f1 = f1_score(y_test_ids, preds, average="macro")

    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1-macro": f1
    })

    print(f"{model_name:30s} | Accuracy: {acc:.4f} | F1-macro: {f1:.4f}")

results_df = pd.DataFrame(results).sort_values("F1-macro", ascending=False)

results_df.to_csv(
    OUTPUT_DIR / "metrikler" / "model_karsilastirma.csv",
    index=False
)

print("\nModel karşılaştırma tablosu:")
print(results_df)



[10] Model karşılaştırması yapılıyor...
TF-IDF + Logistic Regression   | Accuracy: 0.7026 | F1-macro: 0.6723
TF-IDF + Linear SVM            | Accuracy: 0.7054 | F1-macro: 0.6680
Ensemble LR+SVM                | Accuracy: 0.6914 | F1-macro: 0.6639
Önerilen Hibrit Model          | Accuracy: 0.6956 | F1-macro: 0.6664
FinBERT Fine-Tuned             | Accuracy: 0.8600 | F1-macro: 0.8458

Model karşılaştırma tablosu:
                          Model  Accuracy  F1-macro
4            FinBERT Fine-Tuned  0.859978  0.845799
0  TF-IDF + Logistic Regression  0.702627  0.672329
1           TF-IDF + Linear SVM  0.705422  0.668013
3         Önerilen Hibrit Model  0.695640  0.666359
2               Ensemble LR+SVM  0.691448  0.663907


In [20]:
# =============================================================================
# 15. HATA ANALİZİ VE EK DEĞERLENDİRMELER
# =============================================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("\n[11] Hata analizi yapılıyor...")

best_model_name = results_df.iloc[0]["Model"]
best_preds = np.array(model_predictions[best_model_name])

wrong_positions = np.where(best_preds != y_test_ids)[0]

error_rows = []

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

for pos in wrong_positions[:30]:
    error_rows.append({
        "dataset": X_test_reset.loc[pos, "dataset"],
        "text": X_test_reset.loc[pos, "text"],
        "clean_text": X_test_reset.loc[pos, "clean_text"],
        "keywords": X_test_reset.loc[pos, "keyword_text"],
        "true_label": y_test_reset.loc[pos],
        "predicted_label": id2label[int(best_preds[pos])],
        "model": best_model_name,
    })

error_df = pd.DataFrame(error_rows)

error_df.to_csv(
    OUTPUT_DIR / "metrikler" / "hata_analizi.csv",
    index=False
)

print(f"En iyi model: {best_model_name}")
print("\nİlk hata örnekleri:")
print(error_df.head())


# =============================================================================
# 15.1 NORMALIZE CONFUSION MATRIX
# =============================================================================

print("\n[11.1] Normalize edilmiş confusion matrix oluşturuluyor...")

cm_normalized = confusion_matrix(
    y_test_ids,
    best_preds,
    labels=[0, 1, 2],
    normalize="true"
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=VALID_LABELS,
    yticklabels=VALID_LABELS
)
plt.title(f"Normalize Confusion Matrix - {best_model_name}")
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "normalize_confusion_matrix.png",
    dpi=200
)
plt.show()


# =============================================================================
# 15.2 VERİ SETİ BAZLI BAŞARI ANALİZİ
# =============================================================================

print("\n[11.2] Veri seti bazlı başarı analizi yapılıyor...")

dataset_results = []

y_test_ids_reset = np.array(y_test_ids)
best_preds_reset = np.array(best_preds)

for dataset_name in X_test_reset["dataset"].unique():
    mask = X_test_reset["dataset"] == dataset_name

    y_true_dataset = y_test_ids_reset[mask]
    y_pred_dataset = best_preds_reset[mask]

    acc = accuracy_score(y_true_dataset, y_pred_dataset)
    f1 = f1_score(y_true_dataset, y_pred_dataset, average="macro")

    dataset_results.append({
        "Dataset": dataset_name,
        "Accuracy": acc,
        "F1-macro": f1,
        "Sample Count": int(mask.sum())
    })

dataset_results_df = pd.DataFrame(dataset_results)

dataset_results_df.to_csv(
    OUTPUT_DIR / "metrikler" / "veri_seti_bazli_basari.csv",
    index=False
)

print(dataset_results_df)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=dataset_results_df,
    x="F1-macro",
    y="Dataset"
)
plt.title(f"Veri Seti Bazlı F1-macro - {best_model_name}")
plt.xlabel("F1-macro")
plt.ylabel("Veri Seti")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "veri_seti_bazli_f1_macro.png",
    dpi=200
)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(
    data=dataset_results_df,
    x="Accuracy",
    y="Dataset"
)
plt.title(f"Veri Seti Bazlı Accuracy - {best_model_name}")
plt.xlabel("Accuracy")
plt.ylabel("Veri Seti")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "veri_seti_bazli_accuracy.png",
    dpi=200
)
plt.show()


# =============================================================================
# 15.3 CROSS-VALIDATION ANALİZİ
# =============================================================================

print("\n[11.3] Cross-validation analizi yapılıyor...")

cv_models = {}

if "lr" in globals():
    cv_models["Logistic Regression"] = lr

if "svm" in globals():
    cv_models["Linear SVM"] = svm

if "ensemble" in globals():
    cv_models["Ensemble LR+SVM"] = ensemble

if cv_models and "X_train_tfidf" in globals():
    cv_results = []

    for model_name, model in cv_models.items():
        scores = cross_val_score(
            model,
            X_train_tfidf,
            y_train,
            cv=5,
            scoring="f1_macro"
        )

        cv_results.append({
            "Model": model_name,
            "CV Mean F1-macro": scores.mean(),
            "CV Std": scores.std()
        })

    cv_results_df = pd.DataFrame(cv_results)

    cv_results_df.to_csv(
        OUTPUT_DIR / "metrikler" / "cross_validation_sonuclari.csv",
        index=False
    )

    print(cv_results_df)

    plt.figure(figsize=(9, 5))
    sns.barplot(
        data=cv_results_df,
        x="CV Mean F1-macro",
        y="Model"
    )
    plt.title("5-Fold Cross-Validation Sonuçları")
    plt.xlabel("Ortalama F1-macro")
    plt.ylabel("Model")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "grafikler" / "cross_validation_f1_macro.png",
        dpi=200
    )
    plt.show()

else:
    print("Cross-validation atlandı: TF-IDF modelleri hazır değil.")


# =============================================================================
# 15.4 MODEL GÜVEN SKORU ANALİZİ
# =============================================================================

print("\n[11.4] Model güven skoru analizi yapılıyor...")

confidence_df = X_test.reset_index(drop=True).copy()
confidence_df["true_label"] = y_test.reset_index(drop=True)

if best_model_name == "Önerilen Hibrit Model" and "hybrid_model" in globals():
    probabilities = hybrid_model.predict_proba(X_test_hybrid)
    predicted_labels = hybrid_preds

elif best_model_name == "TF-IDF + Logistic Regression" and "lr" in globals():
    probabilities = lr.predict_proba(X_test_tfidf)
    predicted_labels = lr_preds

elif best_model_name == "Ensemble LR+SVM" and "ensemble" in globals() and hasattr(ensemble, "predict_proba"):
    probabilities = ensemble.predict_proba(X_test_tfidf)
    predicted_labels = ens_preds

else:
    probabilities = None
    predicted_labels = [id2label[int(i)] for i in best_preds]

if probabilities is not None:
    confidence_scores = np.max(probabilities, axis=1)

    confidence_df["predicted_label"] = predicted_labels
    confidence_df["confidence_score"] = confidence_scores
    confidence_df["is_correct"] = (
        confidence_df["true_label"] == confidence_df["predicted_label"]
    )

    confidence_df.to_csv(
        OUTPUT_DIR / "metrikler" / "model_guven_skorlari.csv",
        index=False
    )

    print("\nModel güven skoru istatistikleri:")
    print(confidence_df["confidence_score"].describe())

    plt.figure(figsize=(9, 5))
    sns.histplot(
        data=confidence_df,
        x="confidence_score",
        hue="is_correct",
        bins=20,
        kde=True
    )
    plt.title("Model Güven Skoru Dağılımı")
    plt.xlabel("Güven Skoru")
    plt.ylabel("Örnek Sayısı")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "grafikler" / "model_guven_skoru_dagilimi.png",
        dpi=200
    )
    plt.show()

else:
    print(f"Güven skoru analizi atlandı: {best_model_name} için predict_proba hazır değil.")


# =============================================================================
# 15.5 LOGISTIC REGRESSION İÇİN EN ÖNEMLİ TF-IDF KELİMELERİ
# =============================================================================

print("\n[11.5] Logistic Regression için en önemli TF-IDF kelimeleri çıkarılıyor...")

if "lr" in globals() and "baseline_tfidf" in globals():
    feature_names = baseline_tfidf.get_feature_names_out()

    importance_rows = []

    for class_index, class_label in enumerate(lr.classes_):
        coefficients = lr.coef_[class_index]

        top_positive_indices = np.argsort(coefficients)[-15:][::-1]

        for idx in top_positive_indices:
            importance_rows.append({
                "class": class_label,
                "term": feature_names[idx],
                "coefficient": coefficients[idx]
            })

    importance_df = pd.DataFrame(importance_rows)

    importance_df.to_csv(
        OUTPUT_DIR / "metrikler" / "lr_tfidf_onemli_kelimeler.csv",
        index=False
    )

    print(importance_df.head(20))

    for class_label in lr.classes_:
        plot_df = importance_df[
            importance_df["class"] == class_label
        ].sort_values("coefficient", ascending=True)

        plt.figure(figsize=(10, 6))
        sns.barplot(
            data=plot_df,
            x="coefficient",
            y="term"
        )
        plt.title(f"Logistic Regression - {class_label} Sınıfı İçin En Önemli Kelimeler")
        plt.xlabel("Katsayı")
        plt.ylabel("Terim")
        plt.tight_layout()
        plt.savefig(
            OUTPUT_DIR / "grafikler" / f"lr_onemli_kelimeler_{class_label}.png",
            dpi=200
        )
        plt.show()

else:
    print("LR önemli kelimeler analizi atlandı: lr veya baseline_tfidf bulunamadı.")

# =============================================================================
# 15.6 EN ÇOK KARIŞTIRILAN SINIFLAR ANALİZİ
# =============================================================================

print("\n[11.6] En çok karıştırılan sınıflar analizi yapılıyor...")

confusion_rows = []

true_labels_text = [id2label[int(label)] for label in y_test_ids]
pred_labels_text = [id2label[int(label)] for label in best_preds]

for true_label, predicted_label in zip(true_labels_text, pred_labels_text):
    if true_label != predicted_label:
        confusion_rows.append({
            "Gerçek Sınıf": true_label,
            "Tahmin Edilen Sınıf": predicted_label,
            "Hata Türü": f"{true_label} → {predicted_label}"
        })

confusion_errors_df = pd.DataFrame(confusion_rows)

if not confusion_errors_df.empty:
    confusion_summary_df = (
        confusion_errors_df["Hata Türü"]
        .value_counts()
        .reset_index()
    )

    confusion_summary_df.columns = ["Hata Türü", "Hata Sayısı"]

    confusion_summary_df.to_csv(
        OUTPUT_DIR / "metrikler" / "en_cok_karistirilan_siniflar.csv",
        index=False
    )

    print("\nEn çok karıştırılan sınıflar:")
    print(confusion_summary_df)

    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=confusion_summary_df,
        x="Hata Sayısı",
        y="Hata Türü"
    )
    plt.title(f"En Çok Karıştırılan Sınıflar - {best_model_name}")
    plt.xlabel("Hata Sayısı")
    plt.ylabel("Hata Türü")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "grafikler" / "en_cok_karistirilan_siniflar.png",
        dpi=200
    )
    plt.show()

else:
    print("Model test setinde hiç hata yapmadı.")


[11] Hata analizi yapılıyor...
En iyi model: FinBERT Fine-Tuned

İlk hata örnekleri:
                            dataset  \
0  Twitter Financial News Sentiment   
1     FiQA Sentiment Classification   
2  Twitter Financial News Sentiment   
3  Twitter Financial News Sentiment   
4  Twitter Financial News Sentiment   

                                                text  \
0               Canadian National Laying Off Workers   
1  Randgold profit hit by poor gold price but div...   
2  Bristol-Myers Squibb 2020 FactSet EPS consensu...   
3                  Jobless claims hit multi-year low   
4  AMC Entertainment rallies after big Frozen 2 w...   

                                          clean_text  \
0                   canadian national laying workers   
1  randgold profit hit poor gold price dividend s...   
2  bristol myers squibb 2020 factset eps consensu...   
3                  jobless claims hit multi year low   
4       amc entertainment rallies big frozen weekend   

     

In [21]:
# =============================================================================
# 16. MODELİ KENDİ CÜMLELERİNLE TEST ETME
# =============================================================================

print("\n[12] Kendi cümlelerinle test...")


def predict_with_hybrid(text: str):
    clean = clean_text_basic(text)
    kw_pairs = extract_keywords_with_scores(text, top_n=3)
    kw_text = only_keyword_text(kw_pairs)

    vader = sia.polarity_scores(text)["compound"]

    x_main = main_text_vectorizer.transform([clean])

    if keyword_vectorizer is not None:
        x_kw = keyword_vectorizer.transform([kw_text])
    else:
        x_kw = csr_matrix((1, 0))

    x_vader = csr_matrix(
        vader_scaler.transform(
            pd.DataFrame({"vader_score": [vader]})
        )
    )

    x_final = hstack([
        x_main,
        x_kw,
        x_vader
    ])

    pred = hybrid_model.predict(x_final)[0]
    probs = hybrid_model.predict_proba(x_final)[0]

    proba_map = {
        cls: float(prob)
        for cls, prob in zip(hybrid_model.classes_, probs)
    }

    return pred, proba_map, kw_pairs


test_sentences = [
    "The company's revenue increased by 25%.",
    "The stock price collapsed after the scandal.",
    "The market opened flat today.",
    "There are serious concerns about the company's debt.",
    "Despite challenges, the outlook remains positive.",
]

prediction_rows = []

for sent in test_sentences:
    pred, probs, kw_pairs = predict_with_hybrid(sent)

    prediction_rows.append({
        "text": sent,
        "prediction": pred,
        "probabilities": probs,
        "keywords": kw_pairs
    })

    print(f"\nMetin: {sent}")
    print(f"Tahmin: {pred}")
    print(f"Anahtar kelimeler: {kw_pairs}")
    print(f"Olasılıklar: {probs}")

pd.DataFrame(prediction_rows).to_csv(
    OUTPUT_DIR / "tablolar" / "kendi_cumle_testleri.csv",
    index=False
)

# =============================================================================
# 16.1 İNTERAKTİF TAHMİN BÖLÜMÜ
# =============================================================================

print("\n[12.1] İnteraktif tahmin bölümü")

while True:
    user_text = input("Finansal bir cümle giriniz, çıkmak için q yazınız: ")

    if user_text.lower().strip() == "q":
        print("İnteraktif tahmin bölümü sonlandırıldı.")
        break

    pred, probs, kw_pairs = predict_with_hybrid(user_text)

    print("\nGirilen metin:", user_text)
    print("Tahmin:", pred)
    print("Olasılıklar:", probs)
    print("Anahtar kelimeler:", kw_pairs)
    print("-" * 60)


[12] Kendi cümlelerinle test...

Metin: The company's revenue increased by 25%.
Tahmin: positive
Anahtar kelimeler: [('revenue increased', 0.7251), ('company', 0.4044), ('25', 0.2539)]
Olasılıklar: {'negative': 0.023416325782274417, 'neutral': 0.4655968974270777, 'positive': 0.510986776790648}

Metin: The stock price collapsed after the scandal.
Tahmin: negative
Anahtar kelimeler: [('collapsed scandal', 0.7317), ('price collapsed', 0.6466), ('stock price', 0.4669)]
Olasılıklar: {'negative': 0.6352237427497914, 'neutral': 0.20594787796704309, 'positive': 0.1588283792831655}

Metin: The market opened flat today.
Tahmin: negative
Anahtar kelimeler: [('market opened', 0.7303), ('opened flat', 0.6888), ('flat today', 0.6442)]
Olasılıklar: {'negative': 0.4968170410617107, 'neutral': 0.12289938929846793, 'positive': 0.38028356963982135}

Metin: There are serious concerns about the company's debt.
Tahmin: negative
Anahtar kelimeler: [('company debt', 0.7919), ('concerns company', 0.6217), ('c

In [22]:
# =============================================================================
# 17. LIME MODEL AÇIKLAMALARI
# =============================================================================

print("\n[13] LIME model açıklamaları oluşturuluyor...")

RUN_LIME = True

if RUN_LIME:
    try:
        from lime.lime_text import LimeTextExplainer

        class_names = VALID_LABELS

        explainer = LimeTextExplainer(
            class_names=class_names
        )

        def hybrid_predict_proba_for_lime(texts):
            clean_texts = [clean_text_basic(text) for text in texts]

            keyword_texts = []
            vader_scores = []

            for text in texts:
                kw_pairs = extract_keywords_with_scores(text, top_n=3)
                keyword_texts.append(only_keyword_text(kw_pairs))
                vader_scores.append(sia.polarity_scores(str(text))["compound"])

            x_main = main_text_vectorizer.transform(clean_texts)

            if keyword_vectorizer is not None:
                x_kw = keyword_vectorizer.transform(keyword_texts)
            else:
                x_kw = csr_matrix((len(texts), 0))

            x_vader = csr_matrix(
                vader_scaler.transform(
                    pd.DataFrame({"vader_score": vader_scores})
                )
            )

            x_final = hstack([
                x_main,
                x_kw,
                x_vader
            ])

            probs = hybrid_model.predict_proba(x_final)

            ordered_probs = np.zeros((len(texts), len(VALID_LABELS)))

            for i, label in enumerate(VALID_LABELS):
                if label in hybrid_model.classes_:
                    class_index = list(hybrid_model.classes_).index(label)
                    ordered_probs[:, i] = probs[:, class_index]

            return ordered_probs

        lime_sample_text = X_test.reset_index(drop=True).loc[0, "text"]

        explanation = explainer.explain_instance(
            lime_sample_text,
            hybrid_predict_proba_for_lime,
            num_features=10,
            top_labels=1
        )

        lime_html_path = OUTPUT_DIR / "metrikler" / "lime_aciklama_ornegi.html"
        explanation.save_to_file(str(lime_html_path))

        lime_list = explanation.as_list(label=explanation.top_labels[0])

        lime_df = pd.DataFrame(
            lime_list,
            columns=["word", "importance"]
        )

        lime_df.to_csv(
            OUTPUT_DIR / "metrikler" / "lime_aciklama_ornegi.csv",
            index=False
        )

        plt.figure(figsize=(10, 5))
        sns.barplot(
            data=lime_df,
            x="importance",
            y="word"
        )
        plt.title("LIME Kelime Önemleri")
        plt.xlabel("Önem Skoru")
        plt.ylabel("Kelime")
        plt.tight_layout()
        plt.savefig(
            OUTPUT_DIR / "grafikler" / "lime_kelime_onemleri.png",
            dpi=200
        )
        plt.show()

        print("LIME açıklaması kaydedildi:")
        print(lime_html_path)

    except Exception as e:
        print("LIME çalıştırılamadı:", e)
else:
    print("LIME RUN_LIME=False olduğu için atlandı.")


[13] LIME model açıklamaları oluşturuluyor...


/tmp/ipykernel_1631/2798736224.py:84: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 5))


LIME açıklaması kaydedildi:
/content/drive/MyDrive/proje_ciktilari/metrikler/lime_aciklama_ornegi.html


In [23]:
# =============================================================================
# 18. t-SNE GÖRSELLEŞTİRME
# =============================================================================

RUN_TSNE = False

if RUN_TSNE and sentence_model is not None:
    print("\n[14] t-SNE görselleştirme yapılıyor...")

    sample_size = min(300, len(df))
    sample_df = df.sample(sample_size, random_state=RANDOM_STATE).reset_index(drop=True)

    sample_embeddings = sentence_model.encode(
        sample_df["clean_text"].tolist(),
        show_progress_bar=False
    )

    perplexity = min(30, max(5, sample_size // 10))

    tsne = TSNE(
        n_components=2,
        random_state=RANDOM_STATE,
        perplexity=perplexity,
        init="random",
        learning_rate="auto"
    )

    reduced = tsne.fit_transform(sample_embeddings)

    tsne_df = pd.DataFrame({
        "x": reduced[:, 0],
        "y": reduced[:, 1],
        "label": sample_df["label"]
    })

    tsne_df.to_csv(
        OUTPUT_DIR / "tablolar" / "tsne_koordinatlari.csv",
        index=False
    )

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=tsne_df, x="x", y="y", hue="label", alpha=0.75)
    plt.title("Sentence Embedding Space - t-SNE")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "grafikler" / "tsne_embedding.png", dpi=150)
    show_plot()

elif RUN_TSNE and sentence_model is None:
    print("\n[14] t-SNE atlandı: SentenceTransformer yüklü değil veya kullanılmıyor.")
else:
    print("\n[14] t-SNE görselleştirme RUN_TSNE=False olduğu için atlandı.")



[14] t-SNE görselleştirme RUN_TSNE=False olduğu için atlandı.


In [24]:
# =============================================================================
# 19. DETAYLI DEĞERLENDİRME RAPORLARI
# =============================================================================

for model_name, preds in model_predictions.items():
    report = classification_report(
        y_test_ids,
        preds,
        target_names=VALID_LABELS,
        zero_division=0
    )

    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("-", "_")
    )

    with open(
        OUTPUT_DIR / "metrikler" / f"{safe_name}_classification_report.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(report)

    disp = ConfusionMatrixDisplay.from_predictions(
        y_test_ids,
        preds,
        display_labels=VALID_LABELS
    )
    disp.ax_.set_title(model_name)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "grafikler" / f"{safe_name}_confusion_matrix.png",
        dpi=150
    )
    show_plot()

    # =============================================================================
# 19.1 OTOMATİK SONUÇ YORUM RAPORU
# =============================================================================

print("\n[19.1] Otomatik sonuç yorum raporu oluşturuluyor...")

best_row = results_df.iloc[0]
worst_row = results_df.iloc[-1]

report_lines = []

report_lines.append("OTOMATİK MODEL SONUÇ YORUMU")
report_lines.append("=" * 35)
report_lines.append("")
report_lines.append(f"En iyi model: {best_row['Model']}")
report_lines.append(f"En iyi model Accuracy değeri: {best_row['Accuracy']:.4f}")
report_lines.append(f"En iyi model F1-macro değeri: {best_row['F1-macro']:.4f}")
report_lines.append("")
report_lines.append(f"En düşük performans gösteren model: {worst_row['Model']}")
report_lines.append(f"Bu modelin Accuracy değeri: {worst_row['Accuracy']:.4f}")
report_lines.append(f"Bu modelin F1-macro değeri: {worst_row['F1-macro']:.4f}")
report_lines.append("")

if "dataset_results_df" in globals():
    best_dataset_row = dataset_results_df.sort_values(
        "F1-macro",
        ascending=False
    ).iloc[0]

    worst_dataset_row = dataset_results_df.sort_values(
        "F1-macro",
        ascending=True
    ).iloc[0]

    report_lines.append("VERİ SETİ BAZLI DEĞERLENDİRME")
    report_lines.append("-" * 35)
    report_lines.append(
        f"Modelin en başarılı olduğu veri seti: {best_dataset_row['Dataset']} "
        f"(F1-macro: {best_dataset_row['F1-macro']:.4f})"
    )
    report_lines.append(
        f"Modelin en zorlandığı veri seti: {worst_dataset_row['Dataset']} "
        f"(F1-macro: {worst_dataset_row['F1-macro']:.4f})"
    )
    report_lines.append("")

if "confusion_summary_df" in globals() and not confusion_summary_df.empty:
    most_common_error = confusion_summary_df.iloc[0]

    report_lines.append("HATA ANALİZİ")
    report_lines.append("-" * 35)
    report_lines.append(
        f"En sık yapılan hata türü: {most_common_error['Hata Türü']} "
        f"({most_common_error['Hata Sayısı']} kez)"
    )
    report_lines.append("")

if "confidence_df" in globals() and "confidence_score" in confidence_df.columns:
    avg_confidence = confidence_df["confidence_score"].mean()

    report_lines.append("GÜVEN SKORU ANALİZİ")
    report_lines.append("-" * 35)
    report_lines.append(
        f"Modelin ortalama güven skoru: {avg_confidence:.4f}"
    )
    report_lines.append("")

report_lines.append("GENEL YORUM")
report_lines.append("-" * 35)

if best_row["F1-macro"] >= 0.80:
    report_lines.append(
        "Model genel olarak güçlü bir sınıflandırma performansı göstermektedir."
    )
elif best_row["F1-macro"] >= 0.60:
    report_lines.append(
        "Model orta-iyi seviyede performans göstermektedir; hata analizi ile iyileştirilebilir."
    )
else:
    report_lines.append(
        "Model performansı geliştirmeye açıktır; veri dengesi, model seçimi ve özellik çıkarımı iyileştirilebilir."
    )

report_lines.append(
    "F1-macro metriği çok sınıflı duygu analizinde önemlidir çünkü sınıflar arasındaki dengesizliği Accuracy değerine göre daha iyi yansıtır."
)

auto_report_text = "\n".join(report_lines)

with open(
    OUTPUT_DIR / "metrikler" / "otomatik_sonuc_yorum_raporu.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(auto_report_text)

print(auto_report_text)

# =============================================================================
# 19.2 MODEL EĞİTİM SÜRESİ KARŞILAŞTIRMASI
# =============================================================================

print("\n[19.2] Model eğitim süresi karşılaştırması oluşturuluyor...")

training_time_rows = []

if "lr" in globals():
    training_time_rows.append({
        "Model": "Logistic Regression",
        "Yaklaşık Eğitim Süresi": "Kısa",
        "Model Türü": "Klasik ML"
    })

if "svm" in globals():
    training_time_rows.append({
        "Model": "Linear SVM",
        "Yaklaşık Eğitim Süresi": "Orta",
        "Model Türü": "Klasik ML"
    })

if "ensemble" in globals():
    training_time_rows.append({
        "Model": "Ensemble LR+SVM",
        "Yaklaşık Eğitim Süresi": "Orta",
        "Model Türü": "Klasik ML"
    })

if "hybrid_model" in globals():
    training_time_rows.append({
        "Model": "Önerilen Hibrit Model",
        "Yaklaşık Eğitim Süresi": "Orta",
        "Model Türü": "Hibrit ML"
    })

if "bert_preds" in globals() and bert_preds is not None:
    training_time_rows.append({
        "Model": "FinBERT Fine-Tuned",
        "Yaklaşık Eğitim Süresi": "Uzun",
        "Model Türü": "Transformer"
    })

training_time_df = pd.DataFrame(training_time_rows)

training_time_df.to_csv(
    OUTPUT_DIR / "tablolar" / "model_egitim_suresi_karsilastirma.csv",
    index=False
)

print(training_time_df)

# =============================================================================
# 19.3 FİNANSAL TERİM FREKANS ANALİZİ
# =============================================================================

print("\n[19.3] Finansal terim frekans analizi yapılıyor...")

financial_terms = [
    "stock", "market", "share", "shares", "revenue", "profit", "loss",
    "earnings", "growth", "debt", "cash", "investment", "investor",
    "price", "sales", "income", "bank", "company", "quarter", "dividend",
    "bond", "equity", "asset", "liability", "risk", "forecast", "outlook"
]

term_rows = []

for term in financial_terms:
    count = df["clean_text"].str.split().apply(lambda words: words.count(term)).sum()

    term_rows.append({
        "term": term,
        "frequency": int(count)
    })

financial_terms_df = pd.DataFrame(term_rows).sort_values(
    "frequency",
    ascending=False
)

financial_terms_df.to_csv(
    OUTPUT_DIR / "tablolar" / "finansal_terim_frekanslari.csv",
    index=False
)

print(financial_terms_df.head(20))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=financial_terms_df.head(20),
    x="frequency",
    y="term"
)
plt.title("En Sık Geçen Finansal Terimler")
plt.xlabel("Frekans")
plt.ylabel("Finansal Terim")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "finansal_terim_frekanslari.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.4 OTOMATİK MARKDOWN PROJE RAPORU
# =============================================================================

print("\n[19.4] Otomatik markdown proje raporu oluşturuluyor...")

best_row = results_df.iloc[0]

markdown_report = f"""
# Finansal Metinlerde Duygu Analizi Proje Raporu

## 1. Projenin Amacı

Bu projenin amacı, finansal metinleri **negative**, **neutral** ve **positive** sınıflarına ayıran makine öğrenmesi tabanlı bir duygu analizi sistemi geliştirmektir.

## 2. Kullanılan Veri Setleri

- Financial PhraseBank
- Twitter Financial News Sentiment
- FiQA Sentiment Classification

Toplam örnek sayısı: **{len(df)}**

## 3. Kullanılan Yöntemler

Projede TF-IDF, Logistic Regression, Linear SVM, Ensemble model, VADER duygu skoru, KeyBERT anahtar kelime çıkarımı ve FinBERT yaklaşımları kullanılmıştır.

## 4. En İyi Model Sonucu

En iyi model: **{best_row["Model"]}**

Accuracy: **{best_row["Accuracy"]:.4f}**

F1-macro: **{best_row["F1-macro"]:.4f}**

## 5. Değerlendirme

Model başarısı Accuracy ve F1-macro metrikleriyle değerlendirilmiştir. Ayrıca confusion matrix, sınıf bazlı metrikler, hata analizi ve veri seti bazlı başarı analizleri yapılmıştır.

## 6. Ek Analizler

- Normalize confusion matrix oluşturuldu.
- Veri seti bazlı başarı analizi yapıldı.
- En çok karıştırılan sınıflar analiz edildi.
- Finansal terim frekansları çıkarıldı.
- Model sonuçları otomatik olarak yorumlandı.

## 7. Genel Sonuç

Bu proje, finansal metin madenciliği kapsamında çok sınıflı duygu analizi problemini ele almakta ve klasik makine öğrenmesi modelleri ile transformer tabanlı modelleri karşılaştırmaktadır.
"""

with open(
    OUTPUT_DIR / "proje_raporu.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(markdown_report)

print("Markdown rapor oluşturuldu:")
print(OUTPUT_DIR / "proje_raporu.md")

# =============================================================================
# 19.5 MODEL KARŞILAŞTIRMA ISI HARİTASI
# =============================================================================

print("\n[19.5] Model karşılaştırma ısı haritası oluşturuluyor...")

heatmap_df = results_df.set_index("Model")[["Accuracy", "F1-macro"]]

plt.figure(figsize=(8, 5))
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".4f",
    cmap="Blues",
    linewidths=0.5
)
plt.title("Model Karşılaştırma Isı Haritası")
plt.xlabel("Metrik")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "model_karsilastirma_heatmap.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.6 VERİ SETLERİ ARASI BENZERLİK ANALİZİ
# =============================================================================

print("\n[19.6] Veri setleri arası benzerlik analizi yapılıyor...")

dataset_texts = (
    df.groupby("dataset")["clean_text"]
    .apply(lambda texts: " ".join(texts))
)

dataset_names = dataset_texts.index.tolist()

dataset_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

dataset_vectors = dataset_vectorizer.fit_transform(dataset_texts.values)

dataset_similarity = cosine_similarity(dataset_vectors)

dataset_similarity_df = pd.DataFrame(
    dataset_similarity,
    index=dataset_names,
    columns=dataset_names
)

dataset_similarity_df.to_csv(
    OUTPUT_DIR / "tablolar" / "veri_setleri_arasi_benzerlik.csv"
)

print(dataset_similarity_df)

plt.figure(figsize=(7, 6))
sns.heatmap(
    dataset_similarity_df,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    linewidths=0.5
)
plt.title("Veri Setleri Arası TF-IDF Benzerliği")
plt.xlabel("Veri Seti")
plt.ylabel("Veri Seti")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "veri_setleri_arasi_benzerlik.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.7 KELİME BİRLİKTELİK AĞI
# =============================================================================

print("\n[19.7] Kelime birliktelik ağı oluşturuluyor...")

!pip install -q networkx

import networkx as nx
from itertools import combinations
from collections import Counter

top_terms = (
    financial_terms_df
    .sort_values("frequency", ascending=False)
    .head(20)["term"]
    .tolist()
)

cooccurrence_counter = Counter()

for text in df["clean_text"].dropna():
    words = set(str(text).split())
    present_terms = [term for term in top_terms if term in words]

    for pair in combinations(sorted(present_terms), 2):
        cooccurrence_counter[pair] += 1

top_edges = cooccurrence_counter.most_common(30)

G = nx.Graph()

for (term1, term2), weight in top_edges:
    G.add_edge(term1, term2, weight=weight)

plt.figure(figsize=(12, 8))

pos = nx.spring_layout(G, seed=42, k=0.8)

edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
max_weight = max(edge_weights) if edge_weights else 1

edge_widths = [
    1 + (weight / max_weight) * 4
    for weight in edge_weights
]

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=1500,
    alpha=0.9
)

nx.draw_networkx_edges(
    G,
    pos,
    width=edge_widths,
    alpha=0.5
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=10
)

plt.title("Finansal Terimler Kelime Birliktelik Ağı")
plt.axis("off")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "kelime_birliktelik_agi.png",
    dpi=200
)
plt.show()

edge_rows = []

for (term1, term2), weight in top_edges:
    edge_rows.append({
        "term_1": term1,
        "term_2": term2,
        "cooccurrence": weight
    })

cooccurrence_df = pd.DataFrame(edge_rows)

cooccurrence_df.to_csv(
    OUTPUT_DIR / "tablolar" / "kelime_birliktelik_aglari.csv",
    index=False
)

print(cooccurrence_df.head(20))

# =============================================================================
# 19.8 MODEL KARARLILIK ANALİZİ
# =============================================================================

print("\n[19.8] Model kararlılık analizi yapılıyor...")

stability_random_states = [42, 52, 62, 72, 82]

stability_rows = []

for rs in stability_random_states:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=rs,
        stratify=y
    )

    tfidf_s = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2
    )

    X_train_tfidf_s = tfidf_s.fit_transform(X_train_s["clean_text"])
    X_test_tfidf_s = tfidf_s.transform(X_test_s["clean_text"])

    lr_s = LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=rs
    )

    lr_s.fit(X_train_tfidf_s, y_train_s)
    preds_s = lr_s.predict(X_test_tfidf_s)

    acc_s = accuracy_score(y_test_s, preds_s)
    f1_s = f1_score(y_test_s, preds_s, average="macro")

    stability_rows.append({
        "Random State": rs,
        "Accuracy": acc_s,
        "F1-macro": f1_s
    })

stability_df = pd.DataFrame(stability_rows)

stability_df.to_csv(
    OUTPUT_DIR / "metrikler" / "model_kararlilik_analizi.csv",
    index=False
)

print(stability_df)

print("\nKararlılık özeti:")
print(stability_df[["Accuracy", "F1-macro"]].agg(["mean", "std"]))

plt.figure(figsize=(8, 5))
sns.lineplot(
    data=stability_df,
    x="Random State",
    y="F1-macro",
    marker="o"
)
plt.title("Model Kararlılık Analizi - F1-macro")
plt.xlabel("Random State")
plt.ylabel("F1-macro")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "model_kararlilik_f1_macro.png",
    dpi=200
)
plt.show()

plt.figure(figsize=(8, 5))
sns.lineplot(
    data=stability_df,
    x="Random State",
    y="Accuracy",
    marker="o"
)
plt.title("Model Kararlılık Analizi - Accuracy")
plt.xlabel("Random State")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "model_kararlilik_accuracy.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.9 SINIF BAZLI DETAYLI HATA ANALİZİ
# =============================================================================

print("\n[19.9] Sınıf bazlı detaylı hata analizi yapılıyor...")

true_labels_text = [id2label[int(i)] for i in y_test_ids]
pred_labels_text = [id2label[int(i)] for i in best_preds]

class_error_df = pd.DataFrame({
    "true_label": true_labels_text,
    "predicted_label": pred_labels_text
})

class_error_table = pd.crosstab(
    class_error_df["true_label"],
    class_error_df["predicted_label"]
)

class_error_table.to_csv(
    OUTPUT_DIR / "metrikler" / "sinif_bazli_hata_tablosu.csv"
)

print(class_error_table)

plt.figure(figsize=(7, 6))
sns.heatmap(
    class_error_table,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.title("Sınıf Bazlı Hata Analizi")
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "sinif_bazli_hata_analizi.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.10 VERİ SETİ KALİTE ANALİZİ
# =============================================================================

print("\n[19.10] Veri seti kalite analizi yapılıyor...")

dataset_quality_df = df.groupby("dataset").agg(
    sample_count=("text", "count"),
    avg_word_count=("word_count", "mean"),
    median_word_count=("word_count", "median"),
    min_word_count=("word_count", "min"),
    max_word_count=("word_count", "max"),
    std_word_count=("word_count", "std")
).reset_index()

dataset_quality_df.to_csv(
    OUTPUT_DIR / "tablolar" / "veri_seti_kalite_analizi.csv",
    index=False
)

print(dataset_quality_df)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=dataset_quality_df,
    x="avg_word_count",
    y="dataset"
)
plt.title("Veri Setlerine Göre Ortalama Kelime Sayısı")
plt.xlabel("Ortalama Kelime Sayısı")
plt.ylabel("Veri Seti")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "veri_seti_ortalama_kelime_sayisi.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.11 EN ZOR 50 ÖRNEK ANALİZİ
# =============================================================================

print("\n[19.11] En zor 50 örnek analizi yapılıyor...")

if "confidence_df" in globals() and "confidence_score" in confidence_df.columns:
    hardest_samples_df = confidence_df.sort_values(
        "confidence_score",
        ascending=True
    ).head(50)

    hardest_samples_df.to_csv(
        OUTPUT_DIR / "metrikler" / "en_zor_50_ornek.csv",
        index=False
    )

    print(hardest_samples_df[
        ["dataset", "text", "true_label", "predicted_label", "confidence_score"]
    ].head(10))

else:
    hardest_samples_df = error_df.head(50)

    hardest_samples_df.to_csv(
        OUTPUT_DIR / "metrikler" / "en_zor_50_ornek.csv",
        index=False
    )

    print("Güven skoru olmadığı için hatalı tahminlerden ilk 50 örnek kaydedildi.")

    # =============================================================================
# 19.12 FİNANSAL KELİME SÖZLÜĞÜ
# =============================================================================

print("\n[19.12] Finansal kelime sözlüğü oluşturuluyor...")

all_words = " ".join(df["clean_text"].dropna()).split()

word_freq = pd.Series(all_words).value_counts().reset_index()
word_freq.columns = ["term", "frequency"]

financial_dictionary_df = word_freq[
    word_freq["term"].isin(financial_terms)
].sort_values("frequency", ascending=False)

financial_dictionary_df.to_csv(
    OUTPUT_DIR / "tablolar" / "finansal_kelime_sozlugu.csv",
    index=False
)

print(financial_dictionary_df.head(30))

# =============================================================================
# 19.13 WORD2VEC FİNANSAL TERİM ANALİZİ
# =============================================================================

print("\n[19.13] Word2Vec finansal terim analizi yapılıyor...")

!pip install -q gensim

try:
    from gensim.models import Word2Vec

    tokenized_texts = [
        str(text).split()
        for text in df["clean_text"].dropna()
    ]

    word2vec_model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=100,
        window=5,
        min_count=3,
        workers=2,
        seed=RANDOM_STATE
    )

    word2vec_rows = []

    query_terms = [
        "stock", "market", "profit", "loss", "revenue",
        "investment", "investor", "shares", "growth"
    ]

    for term in query_terms:
        if term in word2vec_model.wv:
            similar_words = word2vec_model.wv.most_similar(term, topn=10)

            for similar_term, similarity in similar_words:
                word2vec_rows.append({
                    "query_term": term,
                    "similar_term": similar_term,
                    "similarity": similarity
                })

    word2vec_df = pd.DataFrame(word2vec_rows)

    word2vec_df.to_csv(
        OUTPUT_DIR / "tablolar" / "word2vec_finansal_terim_analizi.csv",
        index=False
    )

    print(word2vec_df.head(30))

except Exception as e:
    print("Word2Vec analizi çalıştırılamadı:", e)

# =============================================================================
# 19.14 FİN BERT VS KLASİK ML KARŞILAŞTIRMA TABLOSU
# =============================================================================

print("\n[19.14] FinBERT ve klasik ML karşılaştırma tablosu oluşturuluyor...")

comparison_rows = []

for _, row in results_df.iterrows():
    model_name = row["Model"]

    if "FinBERT" in model_name:
        model_type = "Transformer tabanlı model"
        training_cost = "Yüksek"
        prediction_cost = "Orta-Yüksek"
        interpretability = "Düşük-Orta"
    elif "Hibrit" in model_name:
        model_type = "Hibrit klasik ML"
        training_cost = "Orta"
        prediction_cost = "Düşük"
        interpretability = "Orta"
    else:
        model_type = "Klasik makine öğrenmesi"
        training_cost = "Düşük"
        prediction_cost = "Düşük"
        interpretability = "Yüksek"

    comparison_rows.append({
        "Model": model_name,
        "Model Türü": model_type,
        "Accuracy": row["Accuracy"],
        "F1-macro": row["F1-macro"],
        "Eğitim Maliyeti": training_cost,
        "Tahmin Maliyeti": prediction_cost,
        "Yorumlanabilirlik": interpretability
    })

model_comparison_extended_df = pd.DataFrame(comparison_rows)

model_comparison_extended_df.to_csv(
    OUTPUT_DIR / "tablolar" / "finbert_klasik_ml_karsilastirma.csv",
    index=False
)

print(model_comparison_extended_df)

# =============================================================================
# 19.15 SHAP ANALİZİ
# =============================================================================

print("\n[19.15] SHAP analizi yapılıyor...")

RUN_SHAP = True

if RUN_SHAP:
    try:
        !pip install -q shap
        import shap

        if "lr" in globals() and "baseline_tfidf" in globals():
            sample_size = min(100, X_test_tfidf.shape[0])

            X_shap = X_test_tfidf[:sample_size]

            explainer = shap.LinearExplainer(
                lr,
                X_train_tfidf,
                feature_perturbation="interventional"
            )

            shap_values = explainer.shap_values(X_shap)

            feature_names = baseline_tfidf.get_feature_names_out()

            shap_summary_rows = []

            for class_idx, class_label in enumerate(lr.classes_):
                class_shap_values = shap_values[class_idx]

                mean_abs_values = np.abs(class_shap_values).mean(axis=0)

                top_indices = np.argsort(mean_abs_values)[-20:][::-1]

                for idx in top_indices:
                    shap_summary_rows.append({
                        "class": class_label,
                        "term": feature_names[idx],
                        "mean_abs_shap": mean_abs_values[idx]
                    })

            shap_summary_df = pd.DataFrame(shap_summary_rows)

            shap_summary_df.to_csv(
                OUTPUT_DIR / "metrikler" / "shap_onemli_kelimeler.csv",
                index=False
            )

            print(shap_summary_df.head(20))

            for class_label in lr.classes_:
                plot_df = shap_summary_df[
                    shap_summary_df["class"] == class_label
                ].sort_values("mean_abs_shap", ascending=True)

                plt.figure(figsize=(10, 6))
                sns.barplot(
                    data=plot_df,
                    x="mean_abs_shap",
                    y="term"
                )
                plt.title(f"SHAP - {class_label} Sınıfı İçin Önemli Kelimeler")
                plt.xlabel("Ortalama Mutlak SHAP Değeri")
                plt.ylabel("Terim")
                plt.tight_layout()
                plt.savefig(
                    OUTPUT_DIR / "grafikler" / f"shap_onemli_kelimeler_{class_label}.png",
                    dpi=200
                )
                plt.show()

        else:
            print("SHAP atlandı: lr veya baseline_tfidf bulunamadı.")

    except Exception as e:
        print("SHAP analizi çalıştırılamadı:", e)

else:
    print("SHAP RUN_SHAP=False olduğu için atlandı.")

    # =============================================================================
# 19.16 SONUÇ DASHBOARD GRAFİĞİ
# =============================================================================

print("\n[19.16] Sonuç dashboard grafiği oluşturuluyor...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.barplot(
    data=results_df,
    x="F1-macro",
    y="Model",
    ax=axes[0, 0]
)
axes[0, 0].set_title("Model F1-macro Karşılaştırması")
axes[0, 0].set_xlim(0, 1)

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=VALID_LABELS,
    yticklabels=VALID_LABELS,
    ax=axes[0, 1]
)
axes[0, 1].set_title("Normalize Confusion Matrix")
axes[0, 1].set_xlabel("Tahmin")
axes[0, 1].set_ylabel("Gerçek")

if "dataset_results_df" in globals():
    sns.barplot(
        data=dataset_results_df,
        x="F1-macro",
        y="Dataset",
        ax=axes[1, 0]
    )
    axes[1, 0].set_title("Veri Seti Bazlı F1-macro")
    axes[1, 0].set_xlim(0, 1)
else:
    axes[1, 0].axis("off")

if "confusion_summary_df" in globals() and not confusion_summary_df.empty:
    sns.barplot(
        data=confusion_summary_df,
        x="Hata Sayısı",
        y="Hata Türü",
        ax=axes[1, 1]
    )
    axes[1, 1].set_title("En Çok Karıştırılan Sınıflar")
else:
    axes[1, 1].axis("off")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "sonuc_dashboard.png",
    dpi=250
)
plt.show()

# =============================================================================
# 19.17 OTOMATİK PDF RAPOR
# =============================================================================

print("\n[19.17] Otomatik PDF rapor oluşturuluyor...")

!pip install -q reportlab

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Image,
        Table,
        TableStyle
    )
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib import colors

    pdf_path = OUTPUT_DIR / "proje_raporu.pdf"

    doc = SimpleDocTemplate(
        str(pdf_path),
        pagesize=A4
    )

    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("Finansal Metinlerde Duygu Analizi Proje Raporu", styles["Title"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph("1. Projenin Amacı", styles["Heading2"]))
    story.append(Paragraph(
        "Bu projenin amacı finansal metinleri negative, neutral ve positive sınıflarına ayıran bir duygu analizi sistemi geliştirmektir.",
        styles["BodyText"]
    ))
    story.append(Spacer(1, 12))

    story.append(Paragraph("2. Model Sonuçları", styles["Heading2"]))

    table_data = [["Model", "Accuracy", "F1-macro"]]

    for _, row in results_df.iterrows():
        table_data.append([
            row["Model"],
            f"{row['Accuracy']:.4f}",
            f"{row['F1-macro']:.4f}"
        ])

    result_table = Table(table_data)
    result_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold")
    ]))

    story.append(result_table)
    story.append(Spacer(1, 12))

    story.append(Paragraph("3. En İyi Model", styles["Heading2"]))
    story.append(Paragraph(
        f"En iyi model {best_model_name} olarak belirlenmiştir.",
        styles["BodyText"]
    ))
    story.append(Spacer(1, 12))

    image_paths = [
        OUTPUT_DIR / "grafikler" / "model_karsilastirma_f1_macro.png",
        OUTPUT_DIR / "grafikler" / "normalize_confusion_matrix.png",
        OUTPUT_DIR / "grafikler" / "veri_seti_bazli_f1_macro.png",
        OUTPUT_DIR / "grafikler" / "en_cok_karistirilan_siniflar.png",
        OUTPUT_DIR / "grafikler" / "sonuc_dashboard.png",
    ]

    story.append(Paragraph("4. Grafikler", styles["Heading2"]))

    for img_path in image_paths:
        if img_path.exists():
            story.append(Image(str(img_path), width=420, height=260))
            story.append(Spacer(1, 12))

    doc.build(story)

    print("PDF rapor oluşturuldu:")
    print(pdf_path)

except Exception as e:
    print("PDF rapor oluşturulamadı:", e)

    # =============================================================================
# 19.18 STREAMLIT ARAYÜZ DOSYASI OLUŞTURMA
# =============================================================================

print("\n[19.18] Streamlit arayüz dosyası oluşturuluyor...")

streamlit_code = '''
import streamlit as st
import joblib
import pandas as pd
import re
from pathlib import Path
from scipy.sparse import hstack, csr_matrix
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon", quiet=True)

BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "proje_ciktilari" / "modeller"

st.title("Finansal Metinlerde Duygu Analizi")
st.write("Bir finansal haber veya cümle giriniz.")

def clean_text_basic(text):
    text = str(text).lower()
    text = re.sub(r"\\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)
    words = text.split()
    return " ".join([w for w in words if len(w) > 1])

try:
    main_text_vectorizer = joblib.load(MODEL_DIR / "hybrid_main_text_vectorizer.joblib")
    keyword_vectorizer = joblib.load(MODEL_DIR / "hybrid_keyword_vectorizer.joblib")
    vader_scaler = joblib.load(MODEL_DIR / "hybrid_vader_scaler.joblib")
    hybrid_model = joblib.load(MODEL_DIR / "onerilen_hibrit_model.joblib")

    sia = SentimentIntensityAnalyzer()

    user_text = st.text_area("Finansal metin:", height=150)

    if st.button("Tahmin Et"):
        clean = clean_text_basic(user_text)
        vader = sia.polarity_scores(user_text)["compound"]

        x_main = main_text_vectorizer.transform([clean])

        if keyword_vectorizer is not None:
            x_kw = keyword_vectorizer.transform([""])
        else:
            x_kw = csr_matrix((1, 0))

        x_vader = csr_matrix(
            vader_scaler.transform(
                pd.DataFrame({"vader_score": [vader]})
            )
        )

        x_final = hstack([x_main, x_kw, x_vader])

        pred = hybrid_model.predict(x_final)[0]
        probs = hybrid_model.predict_proba(x_final)[0]

        st.subheader("Tahmin")
        st.success(pred)

        proba_df = pd.DataFrame({
            "Sınıf": hybrid_model.classes_,
            "Olasılık": probs
        })

        st.bar_chart(proba_df.set_index("Sınıf"))

except Exception as e:
    st.error(f"Model yüklenemedi: {e}")
'''

streamlit_path = OUTPUT_DIR / "streamlit_app.py"

with open(
    streamlit_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(streamlit_code)

print("Streamlit dosyası oluşturuldu:")
print(streamlit_path)
print("Çalıştırmak için:")
print("streamlit run streamlit_app.py")

# =============================================================================
# 19.19 FINANCIAL SENTIMENT INDEX
# =============================================================================

print("\n[19.19] Financial Sentiment Index hesaplanıyor...")

sentiment_counts = df["label"].value_counts(normalize=True).reindex(VALID_LABELS)

negative_ratio = sentiment_counts["negative"]
neutral_ratio = sentiment_counts["neutral"]
positive_ratio = sentiment_counts["positive"]

financial_sentiment_index = positive_ratio - negative_ratio

fsi_df = pd.DataFrame({
    "Sentiment": ["negative", "neutral", "positive", "Financial Sentiment Index"],
    "Value": [
        negative_ratio,
        neutral_ratio,
        positive_ratio,
        financial_sentiment_index
    ]
})

fsi_df.to_csv(
    OUTPUT_DIR / "metrikler" / "financial_sentiment_index.csv",
    index=False
)

print(fsi_df)

plt.figure(figsize=(7, 5))
sns.barplot(
    data=fsi_df[fsi_df["Sentiment"] != "Financial Sentiment Index"],
    x="Sentiment",
    y="Value"
)
plt.title("Financial Sentiment Ratios")
plt.xlabel("Sentiment")
plt.ylabel("Ratio")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "financial_sentiment_ratios.png",
    dpi=200
)
plt.show()

print(f"Financial Sentiment Index: {financial_sentiment_index:.4f}")

# =============================================================================
# 19.20 TOPIC MODELING - LDA
# =============================================================================

print("\n[19.20] Topic Modeling LDA analizi yapılıyor...")

from sklearn.decomposition import LatentDirichletAllocation

TOPIC_COUNT = 5

topic_vectorizer = CountVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=5,
    max_df=0.90
)

X_topics = topic_vectorizer.fit_transform(df["clean_text"])

lda_model = LatentDirichletAllocation(
    n_components=TOPIC_COUNT,
    random_state=RANDOM_STATE,
    learning_method="batch"
)

topic_matrix = lda_model.fit_transform(X_topics)

topic_words = topic_vectorizer.get_feature_names_out()

topic_rows = []

for topic_idx, topic in enumerate(lda_model.components_):
    top_indices = topic.argsort()[-15:][::-1]

    top_terms = [topic_words[i] for i in top_indices]

    topic_rows.append({
        "Topic": f"Topic {topic_idx + 1}",
        "Top Terms": ", ".join(top_terms)
    })

topic_terms_df = pd.DataFrame(topic_rows)

topic_terms_df.to_csv(
    OUTPUT_DIR / "tablolar" / "lda_topic_terms.csv",
    index=False
)

print(topic_terms_df)

df["topic_id"] = topic_matrix.argmax(axis=1)
df["topic"] = df["topic_id"].apply(lambda x: f"Topic {x + 1}")

topic_distribution_df = df["topic"].value_counts().reset_index()
topic_distribution_df.columns = ["Topic", "Document Count"]

topic_distribution_df.to_csv(
    OUTPUT_DIR / "tablolar" / "topic_distribution.csv",
    index=False
)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=topic_distribution_df,
    x="Document Count",
    y="Topic"
)
plt.title("Topic Distribution")
plt.xlabel("Document Count")
plt.ylabel("Topic")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "topic_distribution.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.20 TOPIC MODELING - LDA
# =============================================================================

print("\n[19.20] Topic Modeling LDA analizi yapılıyor...")

from sklearn.decomposition import LatentDirichletAllocation

TOPIC_COUNT = 5

topic_vectorizer = CountVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=5,
    max_df=0.90
)

X_topics = topic_vectorizer.fit_transform(df["clean_text"])

lda_model = LatentDirichletAllocation(
    n_components=TOPIC_COUNT,
    random_state=RANDOM_STATE,
    learning_method="batch"
)

topic_matrix = lda_model.fit_transform(X_topics)

topic_words = topic_vectorizer.get_feature_names_out()

topic_rows = []

for topic_idx, topic in enumerate(lda_model.components_):
    top_indices = topic.argsort()[-15:][::-1]

    top_terms = [topic_words[i] for i in top_indices]

    topic_rows.append({
        "Topic": f"Topic {topic_idx + 1}",
        "Top Terms": ", ".join(top_terms)
    })

topic_terms_df = pd.DataFrame(topic_rows)

topic_terms_df.to_csv(
    OUTPUT_DIR / "tablolar" / "lda_topic_terms.csv",
    index=False
)

print(topic_terms_df)

df["topic_id"] = topic_matrix.argmax(axis=1)
df["topic"] = df["topic_id"].apply(lambda x: f"Topic {x + 1}")

topic_distribution_df = df["topic"].value_counts().reset_index()
topic_distribution_df.columns = ["Topic", "Document Count"]

topic_distribution_df.to_csv(
    OUTPUT_DIR / "tablolar" / "topic_distribution.csv",
    index=False
)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=topic_distribution_df,
    x="Document Count",
    y="Topic"
)
plt.title("Topic Distribution")
plt.xlabel("Document Count")
plt.ylabel("Topic")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "grafikler" / "topic_distribution.png",
    dpi=200
)
plt.show()

# =============================================================================
# 19.22 MCNEMAR TESTİ - FINBERT VS HIBRIT MODEL
# =============================================================================

print("\n[19.22] McNemar testi yapılıyor...")

try:
    from statsmodels.stats.contingency_tables import mcnemar

    if "bert_preds" in globals() and bert_preds is not None and "hybrid_preds" in globals():
        finbert_correct = np.array(bert_preds) == np.array(y_test_ids)
        hybrid_correct = np.array([label2id[p] for p in hybrid_preds]) == np.array(y_test_ids)

        both_correct = np.sum(finbert_correct & hybrid_correct)
        finbert_correct_hybrid_wrong = np.sum(finbert_correct & ~hybrid_correct)
        finbert_wrong_hybrid_correct = np.sum(~finbert_correct & hybrid_correct)
        both_wrong = np.sum(~finbert_correct & ~hybrid_correct)

        mcnemar_table = [
            [both_correct, finbert_correct_hybrid_wrong],
            [finbert_wrong_hybrid_correct, both_wrong]
        ]

        result = mcnemar(
            mcnemar_table,
            exact=False,
            correction=True
        )

        mcnemar_df = pd.DataFrame(
            mcnemar_table,
            index=["FinBERT Correct", "FinBERT Wrong"],
            columns=["Hybrid Correct", "Hybrid Wrong"]
        )

        mcnemar_df.to_csv(
            OUTPUT_DIR / "metrikler" / "mcnemar_finbert_vs_hybrid.csv"
        )

        with open(
            OUTPUT_DIR / "metrikler" / "mcnemar_test_sonucu.txt",
            "w",
            encoding="utf-8"
        ) as f:
            f.write("McNemar Testi: FinBERT vs Hibrit Model\n")
            f.write("=" * 40 + "\n")
            f.write(str(mcnemar_df) + "\n\n")
            f.write(f"Statistic: {result.statistic:.4f}\n")
            f.write(f"p-value: {result.pvalue:.6f}\n")

        print(mcnemar_df)
        print(f"McNemar statistic: {result.statistic:.4f}")
        print(f"p-value: {result.pvalue:.6f}")

    else:
        print("McNemar testi atlandı: FinBERT veya hibrit tahminleri bulunamadı.")

except Exception as e:
    print("McNemar testi çalıştırılamadı:", e)

    # =============================================================================
# 19.23 OTOMATİK POWERPOINT SUNUMU
# =============================================================================

print("\n[19.23] Otomatik PowerPoint sunumu oluşturuluyor...")

!pip install -q python-pptx

try:
    from pptx import Presentation
    from pptx.util import Inches, Pt

    pptx_path = OUTPUT_DIR / "finansal_duygu_analizi_sunumu.pptx"

    prs = Presentation()

    def add_title_slide(title, subtitle):
        slide_layout = prs.slide_layouts[0]
        slide = prs.slides.add_slide(slide_layout)
        slide.shapes.title.text = title
        slide.placeholders[1].text = subtitle

    def add_text_slide(title, bullet_items):
        slide_layout = prs.slide_layouts[1]
        slide = prs.slides.add_slide(slide_layout)
        slide.shapes.title.text = title
        body = slide.placeholders[1].text_frame
        body.clear()

        for item in bullet_items:
            p = body.add_paragraph()
            p.text = item
            p.level = 0
            p.font.size = Pt(18)

    def add_image_slide(title, image_path):
        slide_layout = prs.slide_layouts[5]
        slide = prs.slides.add_slide(slide_layout)
        slide.shapes.title.text = title

        if Path(image_path).exists():
            slide.shapes.add_picture(
                str(image_path),
                Inches(0.7),
                Inches(1.4),
                width=Inches(8.5)
            )

    add_title_slide(
        "Finansal Metinlerde Duygu Analizi",
        "Negative / Neutral / Positive sınıflandırma projesi"
    )

    add_text_slide(
        "Projenin Amacı",
        [
            "Finansal metinleri üç duygu sınıfına ayırmak",
            "Klasik makine öğrenmesi ve FinBERT modellerini karşılaştırmak",
            "Model başarılarını görselleştirmek ve yorumlamak"
        ]
    )

    add_text_slide(
        "Kullanılan Veri Setleri",
        [
            "Financial PhraseBank",
            "Twitter Financial News Sentiment",
            "FiQA Sentiment Classification",
            f"Toplam örnek sayısı: {len(df)}"
        ]
    )

    add_image_slide(
        "Sınıf Dağılımı",
        OUTPUT_DIR / "grafikler" / "sinif_dagilimi.png"
    )

    add_image_slide(
        "Model Karşılaştırması",
        OUTPUT_DIR / "grafikler" / "model_karsilastirma_f1_macro.png"
    )

    add_image_slide(
        "Normalize Confusion Matrix",
        OUTPUT_DIR / "grafikler" / "normalize_confusion_matrix.png"
    )

    add_image_slide(
        "Veri Seti Bazlı Başarı",
        OUTPUT_DIR / "grafikler" / "veri_seti_bazli_f1_macro.png"
    )

    add_image_slide(
        "Sonuç Dashboard",
        OUTPUT_DIR / "grafikler" / "sonuc_dashboard.png"
    )

    add_text_slide(
        "Sonuç",
        [
            f"En iyi model: {best_model_name}",
            "Accuracy ve F1-macro metrikleri ile değerlendirme yapıldı",
            "Hata analizi, SHAP, LIME, Word2Vec ve Topic Modeling eklendi",
            "Tüm çıktılar Google Drive proje klasörüne kaydedildi"
        ]
    )

    prs.save(pptx_path)

    print("PowerPoint sunumu oluşturuldu:")
    print(pptx_path)

except Exception as e:
    print("PowerPoint sunumu oluşturulamadı:", e)

# =============================================================================
# 19.24 OTOMATİK DOCX AKADEMİK RAPOR
# =============================================================================

print("\n[19.24] Otomatik DOCX akademik rapor oluşturuluyor...")

!pip install -q python-docx

try:
    from docx import Document
    from docx.shared import Inches

    docx_path = OUTPUT_DIR / "finansal_duygu_analizi_akademik_rapor.docx"

    document = Document()

    document.add_heading("Finansal Metinlerde Duygu Analizi", 0)

    document.add_heading("Abstract", level=1)
    document.add_paragraph(
        "This project aims to classify financial texts into negative, neutral, and positive sentiment classes. "
        "Classical machine learning models and transformer-based FinBERT were compared using Accuracy and F1-macro metrics."
    )

    document.add_heading("1. Introduction", level=1)
    document.add_paragraph(
        "Financial sentiment analysis is an important natural language processing task used to understand market-related texts, "
        "financial news, and investor-oriented statements."
    )

    document.add_heading("2. Dataset", level=1)
    document.add_paragraph(
        "The project uses a combined financial sentiment dataset consisting of Financial PhraseBank, "
        "Twitter Financial News Sentiment, and FiQA Sentiment Classification datasets."
    )

    document.add_paragraph(f"Total number of samples: {len(df)}")

    document.add_heading("3. Methodology", level=1)
    document.add_paragraph(
        "The methodology includes text preprocessing, TF-IDF feature extraction, VADER sentiment scoring, "
        "KeyBERT keyword extraction, Logistic Regression, Linear SVM, Ensemble learning, Hybrid modeling, and FinBERT fine-tuning."
    )

    document.add_heading("4. Results", level=1)

    table = document.add_table(rows=1, cols=3)
    table.style = "Table Grid"

    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = "Model"
    hdr_cells[1].text = "Accuracy"
    hdr_cells[2].text = "F1-macro"

    for _, row in results_df.iterrows():
        row_cells = table.add_row().cells
        row_cells[0].text = str(row["Model"])
        row_cells[1].text = f"{row['Accuracy']:.4f}"
        row_cells[2].text = f"{row['F1-macro']:.4f}"

    document.add_paragraph(f"The best performing model is: {best_model_name}")

    image_paths = [
        OUTPUT_DIR / "grafikler" / "model_karsilastirma_f1_macro.png",
        OUTPUT_DIR / "grafikler" / "normalize_confusion_matrix.png",
        OUTPUT_DIR / "grafikler" / "veri_seti_bazli_f1_macro.png",
        OUTPUT_DIR / "grafikler" / "sentiment_by_topic.png",
    ]

    document.add_heading("5. Visual Results", level=1)

    for img_path in image_paths:
        if img_path.exists():
            document.add_picture(str(img_path), width=Inches(5.8))

    document.add_heading("6. Discussion", level=1)
    document.add_paragraph(
        "The comparison shows that transformer-based models can provide strong performance, while classical ML models remain interpretable and computationally efficient. "
        "Additional analyses such as LIME, SHAP, topic modeling, and error analysis help explain model behavior."
    )

    document.add_heading("7. Conclusion", level=1)
    document.add_paragraph(
        "The project provides a comprehensive financial sentiment analysis pipeline, including data preparation, model training, evaluation, visualization, and automated reporting."
    )

    document.save(docx_path)

    print("DOCX akademik rapor oluşturuldu:")
    print(docx_path)

except Exception as e:
    print("DOCX rapor oluşturulamadı:", e)

# =============================================================================
# 19.25 ROC-AUC ANALİZİ
# =============================================================================

print("\n[19.25] ROC-AUC analizi yapılıyor...")

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

try:
    y_test_bin = label_binarize(
        y_test_ids,
        classes=[0, 1, 2]
    )

    roc_available = False

    if best_model_name == "Önerilen Hibrit Model" and "hybrid_model" in globals():
        y_score = hybrid_model.predict_proba(X_test_hybrid)
        roc_available = True

    elif best_model_name == "TF-IDF + Logistic Regression" and "lr" in globals():
        y_score = lr.predict_proba(X_test_tfidf)
        roc_available = True

    elif best_model_name == "Ensemble LR+SVM" and "ensemble" in globals() and hasattr(ensemble, "predict_proba"):
        y_score = ensemble.predict_proba(X_test_tfidf)
        roc_available = True

    elif best_model_name == "FinBERT Fine-Tuned" and "bert_probabilities" in globals():
        y_score = bert_probabilities
        roc_available = True

    if roc_available:
        roc_rows = []

        plt.figure(figsize=(8, 6))

        for i, label in enumerate(VALID_LABELS):
            fpr, tpr, _ = roc_curve(
                y_test_bin[:, i],
                y_score[:, i]
            )

            roc_auc = auc(fpr, tpr)

            roc_rows.append({
                "class": label,
                "auc": roc_auc
            })

            plt.plot(
                fpr,
                tpr,
                label=f"{label} AUC = {roc_auc:.3f}"
            )

        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.title(f"ROC-AUC Curve - {best_model_name}")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            OUTPUT_DIR / "grafikler" / "roc_auc_curve.png",
            dpi=200
        )
        plt.show()

        roc_auc_df = pd.DataFrame(roc_rows)

        roc_auc_df.to_csv(
            OUTPUT_DIR / "metrikler" / "roc_auc_sonuclari.csv",
            index=False
        )

        print(roc_auc_df)

    else:
        print("ROC-AUC atlandı: En iyi model için predict_proba veya olasılık skorları bulunamadı.")

except Exception as e:
    print("ROC-AUC analizi çalıştırılamadı:", e)


[19.1] Otomatik sonuç yorum raporu oluşturuluyor...
OTOMATİK MODEL SONUÇ YORUMU

En iyi model: FinBERT Fine-Tuned
En iyi model Accuracy değeri: 0.8600
En iyi model F1-macro değeri: 0.8458

En düşük performans gösteren model: Ensemble LR+SVM
Bu modelin Accuracy değeri: 0.6914
Bu modelin F1-macro değeri: 0.6639

VERİ SETİ BAZLI DEĞERLENDİRME
-----------------------------------
Modelin en başarılı olduğu veri seti: Financial PhraseBank (F1-macro: 0.8688)
Modelin en zorlandığı veri seti: FiQA Sentiment Classification (F1-macro: 0.4986)

HATA ANALİZİ
-----------------------------------
En sık yapılan hata türü: neutral → positive (150 kez)

GENEL YORUM
-----------------------------------
Model genel olarak güçlü bir sınıflandırma performansı göstermektedir.
F1-macro metriği çok sınıflı duygu analizinde önemlidir çünkü sınıflar arasındaki dengesizliği Accuracy değerine göre daha iyi yansıtır.

[19.2] Model eğitim süresi karşılaştırması oluşturuluyor...
                   Model Yaklaşık Eğit

In [25]:
# =============================================================================
# STREAMLIT BAŞLAT - NGROK YOK, LOCALTUNNEL VAR
# =============================================================================

!pip install -q streamlit
!npm install -g localtunnel

import subprocess
import time

streamlit_path = "/content/drive/MyDrive/proje_ciktilari/streamlit_app.py"

subprocess.Popen([
    "streamlit",
    "run",
    streamlit_path,
    "--server.port",
    "8501",
    "--server.headless",
    "true"
])

time.sleep(8)

lt_process = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

print("Aşağıda LocalTunnel linki çıkacak:")
for _ in range(10):
    line = lt_process.stdout.readline()
    if line:
        print(line.strip())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 11.16.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.16.0
npm notice To update run: npm install -g npm@11.16.0
npm notice
⠙Aşağıda LocalTunnel linki çıkacak:
your url is: https://fresh-carpets-write.loca.lt


KeyboardInterrupt: 

In [26]:
# =============================================================================
# 20. PROJE ÖZETİ
# =============================================================================

summary = f"""
PROJE ÖZETİ
===========

Veri seti: Combined Financial Sentiment Dataset

Kullanılan veri setleri:
1. Financial PhraseBank
2. Twitter Financial News Sentiment
3. FiQA Sentiment Classification

Amaç:
Finansal metinleri negative, neutral ve positive duygu sınıflarına ayıran kapsamlı bir duygu analizi sistemi geliştirmek ve klasik makine öğrenmesi yöntemleri ile transformer tabanlı modelleri karşılaştırmak.

Toplam örnek sayısı: {len(df)}

Sınıf dağılımı:
{df["label"].value_counts().reindex(VALID_LABELS).to_string()}

Veri setlerine göre dağılım:
{df["dataset"].value_counts().to_string()}

Kelime uzunluğu p99: {p99_words}
FinBERT max_length: {MAX_LENGTH}

Uygulanan işlemler:

1. Üç farklı finansal duygu analizi veri seti birleştirildi.
2. Veri setleri ortak veri formatına dönüştürüldü.
3. FiQA skorları duygu etiketlerine çevrildi.
4. Farklı veri setlerindeki etiketler standartlaştırıldı.
5. Eksik, boş ve tekrar eden kayıtlar temizlendi.
6. Veri seti ve sınıf dağılımları analiz edildi.
7. Kelime uzunluğu ve veri kalitesi analizleri gerçekleştirildi.
8. Metin temizleme ve ön işleme adımları uygulandı.
9. N-gram analizleri gerçekleştirildi.
10. Kelime bulutları oluşturuldu.
11. Sınıf bazlı TF-IDF analizleri gerçekleştirildi.
12. KeyBERT ile anahtar kelime çıkarımı yapıldı.
13. VADER duygu skorları hesaplandı.
14. Eğitim ve test veri setleri oluşturuldu.
15. TF-IDF + Logistic Regression modeli eğitildi.
16. TF-IDF + Linear SVM modeli eğitildi.
17. Ensemble (LR + SVM) modeli geliştirildi.
18. TF-IDF + KeyBERT + VADER tabanlı hibrit model geliştirildi.
19. FinBERT modeli fine-tune edilerek eğitildi.
20. Modeller Accuracy ve F1-macro metrikleri ile değerlendirildi.
21. En iyi model seçildi ve hata analizi gerçekleştirildi.
22. Normalize edilmiş Confusion Matrix oluşturuldu.
23. Veri seti bazlı başarı analizleri gerçekleştirildi.
24. 5-Fold Cross Validation uygulandı.
25. Model güven skoru analizi gerçekleştirildi.
26. Önemli TF-IDF terimleri çıkarıldı.
27. En çok karıştırılan sınıflar analiz edildi.
28. Sınıf bazlı detaylı hata analizi gerçekleştirildi.
29. Kendi finansal örnek cümleleri ile model test edildi.
30. Cümle benzerliği analizi gerçekleştirildi.
31. TF-IDF tabanlı otomatik özetleme uygulandı.
32. LIME ile model açıklanabilirliği sağlandı.
33. SHAP ile özellik önem analizi gerçekleştirildi.
34. t-SNE analizi opsiyonel hale getirildi.
35. Model performansları grafiklerle görselleştirildi.
36. Model karşılaştırma ısı haritaları oluşturuldu.
37. Veri setleri arası TF-IDF benzerlik analizi yapıldı.
38. Finansal terimler için kelime birliktelik ağı oluşturuldu.
39. Finansal terim frekans analizi gerçekleştirildi.
40. Finansal kelime sözlüğü oluşturuldu.
41. Word2Vec ile finansal terim benzerlikleri analiz edildi.
42. Model kararlılık analizi gerçekleştirildi.
43. En zor 50 örnek analiz edildi.
44. FinBERT ve klasik makine öğrenmesi modelleri karşılaştırıldı.
45. Otomatik sonuç yorum raporu oluşturuldu.
46. Markdown raporu oluşturuldu.
47. PDF proje raporu oluşturuldu.
48. Dashboard görselleştirmesi oluşturuldu.
49. Streamlit tabanlı kullanıcı arayüzü geliştirildi.
50. Financial Sentiment Index hesaplandı.
51. LDA ile Topic Modeling gerçekleştirildi.
52. Topic bazlı duygu dağılımları analiz edildi.
53. FinBERT ve Hibrit Model McNemar testi ile karşılaştırıldı.
54. PowerPoint sunumu otomatik oluşturuldu.
55. DOCX formatında akademik rapor üretildi.
56. Tüm grafikler, tablolar, metrikler ve modeller çıktı klasörüne kaydedildi.
57. ROC-AUC analizi ile sınıfların ayrılabilirliği değerlendirildi.

Önerilen Hibrit Model:
TF-IDF(clean_text)
+ TF-IDF(KeyBERT keywords)
+ VADER compound score
+ Logistic Regression

En İyi Model:
{best_model_name}

Üretilen Çıktılar:
- Normalize Confusion Matrix
- Veri Seti Bazlı Başarı Analizi
- SHAP Analizi
- LIME Analizi
- Word2Vec Finansal Terim Analizi
- Kelime Birliktelik Ağı
- Veri Setleri Arası Benzerlik Analizi
- Topic Modeling Sonuçları
- Sentiment by Topic Analizi
- Dashboard Görselleştirmesi
- PDF Proje Raporu
- DOCX Akademik Rapor
- PowerPoint Sunumu
- Streamlit Arayüzü

Sonuç:
Bu proje, finansal metinlerde çok sınıflı duygu analizi için veri hazırlama, özellik çıkarımı, klasik makine öğrenmesi, transformer tabanlı modeller, açıklanabilir yapay zeka, konu modelleme, hata analizi, otomatik raporlama ve görselleştirme bileşenlerini bir araya getiren kapsamlı bir finansal metin madenciliği sistemi sunmaktadır.
"""

with open(
    OUTPUT_DIR / "proje_ozeti.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(summary)

print(summary)

print("\nProje tamamlandı. Tüm çıktılar şu klasöre kaydedildi:")
print(OUTPUT_DIR)


PROJE ÖZETİ

Veri seti: Combined Financial Sentiment Dataset

Kullanılan veri setleri:
1. Financial PhraseBank
2. Twitter Financial News Sentiment
3. FiQA Sentiment Classification

Amaç:
Finansal metinleri negative, neutral ve positive duygu sınıflarına ayıran kapsamlı bir duygu analizi sistemi geliştirmek ve klasik makine öğrenmesi yöntemleri ile transformer tabanlı modelleri karşılaştırmak.

Toplam örnek sayısı: 17889

Sınıf dağılımı:
label
negative    2776
neutral     5283
positive    9830

Veri setlerine göre dağılım:
dataset
Twitter Financial News Sentiment    11927
Financial PhraseBank                 4840
FiQA Sentiment Classification        1122

Kelime uzunluğu p99: 44
FinBERT max_length: 64

Uygulanan işlemler:

1. Üç farklı finansal duygu analizi veri seti birleştirildi.
2. Veri setleri ortak veri formatına dönüştürüldü.
3. FiQA skorları duygu etiketlerine çevrildi.
4. Farklı veri setlerindeki etiketler standartlaştırıldı.
5. Eksik, boş ve tekrar eden kayıtlar temizlendi.
6